# CIC-IDS2017 Research Notebook — v3 GPU & Cross-Dataset Evaluation[¶](#CIC-IDS2017-Research-Notebook)

## Staged pipeline search, model tuning, and explainable feature analysis[¶](#Staged-pipeline-search,-model-tuning,-and-explainable-feature-analysis)

**Study type:** empirical comparative study for binary network intrusion detection, with an optional external generalization test on CSE-CIC-IDS2018.

**Working objective:** systematically compare defensible, leakage-safe data-preparation and model configurations for Logistic Regression, Random Forest, and XGBoost; tune only promising complete pipelines; explain a selected tree model with SHAP; and test whether SHAP-guided feature reduction preserves detection performance.

**Important scope statement:** this notebook seeks the strongest *defensible configuration under the stated dataset, split protocol, metric, search budget, and hardware constraints*. It does not claim to discover a universally best preprocessing method or a universally best NIDS model.

### Research questions[¶](#Research-questions)

- **RQ1:** How do selected preprocessing and model configurations affect binary intrusion-detection performance on CIC-IDS2017?
- **RQ2:** How do class weighting and training-only resampling affect the Benign/Attack trade-off?
- **RQ3:** Which flow features contribute most to the selected tree model's predictions?
- **RQ4:** Can a smaller SHAP-ranked feature subset preserve useful detection performance?

### Research boundary[¶](#Research-boundary)

This is an offline experimental paper. Deployment, real-time serving, CI/CD, production monitoring, and model governance are not experimental variables. UV, Git/GitHub, and DVC are used for reproducibility, not to claim a new MLOps contribution.

## GPU and estimator scope

- **XGBoost:** configured with `tree_method="hist"` and `device="cuda"`; the notebook runs a CUDA smoke test and stops if CUDA is not actually active. It does not silently fall back to CPU.
- **Logistic Regression and Random Forest:** the notebook uses scikit-learn implementations, which run on CPU. GPU alternatives require RAPIDS cuML. NVIDIA currently supports RAPIDS on Linux and Windows through WSL2, not as a native Windows package; installation also needs a compatible CUDA/driver combination. See the official [XGBoost GPU guide](https://xgboost.readthedocs.io/en/stable/gpu/) and [RAPIDS platform support](https://docs.nvidia.com/datascience/platform-support/index.html).
- GPU acceleration applies to XGBoost training and prediction, not pandas loading, scikit-learn preprocessing, Logistic Regression, Random Forest, or every grid-search step.

The GPU smoke test is intentionally a hard check. If it fails, the notebook tells you what to fix rather than reporting CPU results as GPU results.


For the standard Windows x86-64 XGBoost wheel, install/update through the active project environment with:

```powershell
uv add "xgboost>=2.0"
```

Do not install the CPU-only `xgboost-cpu` package. The notebook verifies the actual CUDA device before starting the expensive search.

## 1. Complete experimental workflow[¶](#1.-Complete-experimental-workflow)

```
flowchart TD
 A["Eight original CIC-IDS2017 CSV files"] --> B["Ingestion, schema and provenance audit"]
 B --> C["EDA and data-quality audit"]
 C --> D["Deterministic corrections + documented data decisions"]
 D --> E["Choose split from generalization question"]
 E --> F["Train / validation / untouched test"]
 F --> G["Training-only cross-validation"]
 G --> H["Level 0: default baseline pipelines"]
 H --> I["Level 1: one-family-at-a-time pipeline GridSearchCV"]
 I --> J["Level 2: combine promising factors; validate finalists"]
 J --> K["Level 3: model hyperparameter GridSearchCV"]
 K --> L["Validation selection"]
 L --> M["Refit selected pipeline on train + validation"]
 M --> N["Final test evaluation once"]
 L --> O["SHAP from training data"]
 O --> P["Top-k feature subsets; validate on validation set"]
 P --> Q["Refit reduced feature pipeline on train + validation"]
 Q --> N
 N --> R["Metrics, error analysis, cost and limitations"]
```

### The two searches are different[¶](#The-two-searches-are-different)

1. **Pipeline-level search:** compare candidate choices for imputation, outlier treatment, categorical handling, scaling where model-relevant, feature selection, and imbalance handling. Change one meaningful family at a time while keeping the rest fixed.
2. **Model-level search:** after choosing a promising complete pipeline, tune that model's hyperparameters using a compact `GridSearchCV` space.

The experimental unit is the **complete pipeline**, not just the estimator. We do not brute-force every possible Cartesian product. We establish a baseline, run controlled factor searches, combine a small number of defensible candidates, tune finalists, and reserve the test set for final reporting.

## 2. What from the supplied 36-page ML-pipeline framework applies to this paper?[¶](#2.-What-from-the-supplied-36-page-ML-pipeline-framework-applies-to-this-paper?)

The uploaded document is a useful catalogue of the end-to-end ML lifecycle. Its main pillars—problem framing, data collection, EDA, cleaning, feature engineering, split/CV design, model selection/training, tuning, evaluation, and explainability—directly inform this study. Deployment, live monitoring, CI/CD, and production governance are valid ML lifecycle topics, but they are outside this offline research experiment. See the supplied document's end-to-end pillars and minimal order. (supplied PDF, pp. 1–2 and 9).

| PDF pillar | How it is handled in this CIC-IDS2017 study |
| --- | --- |
| Problem framing | Binary flow classification: `BENIGN=0`, every other non-empty attack label = `1`. Keep original attack-family labels for descriptive analysis. |
| Data collection / ingestion | Inventory eight CSVs, verify schema consistency, record source file and dataset version. |
| EDA | Shapes, class distributions, feature types, missingness, infinities, duplicates, correlations, skew and range summaries. |
| Cleaning / preprocessing | Numeric imputation, Protocol handling, train-fitted clipping/transformations and model-specific scaling. |
| Feature engineering / selection | Start with documented CICFlowMeter features; test selected filter methods; use SHAP after fitting for a separate reduction experiment. |
| Splitting / CV | Main split is stratified random unless the research question calls for a group or chronological evaluation. CV must match the chosen split logic. |
| Imbalance | Compare natural distribution, model-specific class weights, random oversampling and random undersampling. |
| Model selection / training | Logistic Regression, Random Forest and XGBoost; default baselines first. |
| Hyperparameter tuning | Compact `GridSearchCV` for discrete shortlisted settings. Use Randomized/Bayesian search only if a wider search later becomes justified. |
| Evaluation | Attack precision/recall/F1, Macro-F1, balanced accuracy, PR-AUC, ROC-AUC, FPR/FNR, MCC and confusion matrices. |
| Explainability | SHAP global/local analysis and validation of reduced feature sets. |
| Deployment / monitoring / governance | Not part of this paper; keep as future engineering work. |

### Non-negotiable methodology rules from the framework[¶](#Non-negotiable-methodology-rules-from-the-framework)

- Split before fitting learned transformations; imputation, scaling, encoding categories, outlier caps, feature selection and resampling belong inside the CV training fold. (supplied PDF, pp. 11–12).
- Choose random, stratified, group or time splitting based on the data structure and intended generalization question—not whichever split produces the highest score. (supplied PDF, pp. 19–20).
- Compare a small number of complete pipeline candidates, then tune model hyperparameters. Avoid an enormous exhaustive search. (supplied PDF, pp. 27–29).
- Treat the test set as untouched until final reporting. Do not use it to pick preprocessing, features, thresholds, or hyperparameters.

## 2A. Full method catalogue: available methods versus methods suitable for this dataset[¶](#2A.-Full-method-catalogue:-available-methods-versus-methods-suitable-for-this-dataset)

This section preserves the method families covered by the supplied PDF but distinguishes **(a) methods that are actually searched in this study, (b) methods retained as optional follow-up experiments, and (c) methods that do not fit the data/task**. A method being listed does not mean it should automatically be applied.

| Family | Methods in the general ML toolkit | CIC-IDS2017 decision | Search / implementation status |
| --- | --- | --- | --- |
| Missing values | Drop rows, drop columns, mean, median, mode, constant, KNN, iterative/model-based imputation, missingness indicators | Audit first. For numeric flow features, compare median, mean, constant-zero, and indicator variants. For Protocol, use most-frequent or `Missing` only if values are missing. | Median/mean/constant and indicator are searched. KNN/Iterative are deferred: expensive at this row count and only justified if missingness is substantial and simple methods are inadequate. Drop-row/column is decided by data-quality evidence, not a blind grid. |
| Outliers | Domain correction, removal, capping/winsorizing, log/square-root/Box-Cox/Yeo-Johnson, robust scaling, binning, Isolation Forest/LOF | Network-flow extremes can be genuine attacks. Correct impossible values only with defensible domain rules; do not delete rare rows just because they are extreme. | No treatment versus train-fitted quantile clipping is searched. Yeo-Johnson is a separate distribution-transform candidate for Logistic Regression. Isolation Forest/LOF are not used as generic outlier cleaners. |
| Duplicates | Exact-row, key/entity duplicates, near duplicates, feature duplicates, conflicting-label duplicates | Audit exact rows, identical feature vectors, and feature vectors associated with conflicting labels. Duplicate groups can be kept together in a group split. | Audit always; removal is off by default. Duplicate-grouped split is a separate evaluation protocol, not a tuning option. |
| Data types | Numeric strings, dates, booleans, categories stored as integers, IDs stored as numeric, mixed missing markers | Convert flow statistics to numeric; treat `Protocol` as nominal categorical; exclude Flow ID/IP/Timestamp from predictors but retain timestamp for time splitting/auditing. | Implemented in data preparation. |
| Categorical encoding | One-hot, ordinal, label, target, frequency, hashing, binary, embeddings/native category handling | Protocol has low cardinality and no natural order. One-hot is appropriate; dropping Protocol is a feature ablation. Ordinal/label encoding would invent order; target/frequency/hash/embeddings solve high-cardinality cases we do not have here. | One-hot versus drop-ablation searched. Other encoders documented but not searched because they are not justified for Protocol. (supplied PDF, pp. 15–16). |
| Numeric scaling | None, standard/Z-score, min-max, robust, max-abs | Logistic Regression is scale-sensitive; tree models are generally not. Robust scaling is not the same as removing outliers. | For Logistic Regression: none, StandardScaler, RobustScaler, MinMaxScaler. Tree models: no scaling by default. (supplied PDF, pp. 16–17). |
| Numeric transformations | `log1p`, square root, Box-Cox, Yeo-Johnson, quantile transform | Choose only after inspecting skew/range. Box-Cox/log require positive data; Yeo-Johnson can handle negative values. | Yeo-Johnson candidate for Logistic Regression. `log1p` may be added for specifically non-negative, heavily skewed columns after EDA; not applied blindly to every column. |
| Feature engineering | Domain ratios, flow rates, interaction/polynomial features, aggregations, time/lag/rolling features, embeddings | CICFlowMeter already contains many flow-level rates/statistics. Avoid duplicating them without a domain rationale. Do not use timestamp as a predictive shortcut in the primary model. | Existing features first. Optional, separately justified ratio/interaction ablation only if feature definitions are understood. No blind polynomial expansion. |
| Feature selection | Variance threshold, correlation filter, ANOVA/chi-square/mutual information, L1/Lasso, RFE/wrapper, tree embedded importance, permutation importance, SHAP | Start with no selection; test variance filtering and a small set of univariate filters inside CV. SHAP selection is a separate post-fit experiment. | Passthrough, VarianceThreshold, ANOVA SelectKBest and optional mutual-information SelectKBest. RFE/L1/embedded/permutation are documented follow-ups, not in the default search. SHAP top-k runs after model selection. (supplied PDF, pp. 18–19). |
| Dimensionality reduction | PCA, Truncated SVD, LDA, t-SNE, UMAP, autoencoders, feature hashing | PCA may be useful for compression but makes original-feature interpretation harder. t-SNE/UMAP are primarily visualization; hashing/autoencoders are unnecessary for this first tabular study. | Not in primary classifier search. May be a separately labelled exploratory visualization/compression study. (supplied PDF, pp. 19). |
| Splitting | Random, stratified, time-based, group-based, holdout | Main stratified random split estimates row-level IID performance. Duplicate-grouped and chronological splits are separate robustness/generalization experiments. | User chooses split before modelling. Never select split strategy by highest score. |
| Cross-validation | K-fold, StratifiedKFold, GroupKFold/StratifiedGroupKFold, TimeSeriesSplit, repeated CV | Use stratified folds for the main imbalanced classification benchmark; group/time CV only when corresponding holdout protocol is used. | CV strategy follows `SPLIT_STRATEGY`. Test set remains untouched. |
| Imbalance | No action, class weights, sample weights, random over/undersampling, SMOTE/SMOTENC, ADASYN, Tomek links, threshold tuning, cost-sensitive learning, anomaly detection | Start with natural distribution and class weights. Random samplers are comparison candidates. Ordinary SMOTE after one-hot encoding can generate fractional Protocol indicators; use a dedicated categorical-aware SMOTENC design if later justified. | None, model-appropriate class weighting, RandomOverSampler and RandomUnderSampler searched. SMOTE/ADASYN/Tomek/anomaly detection/threshold optimization deferred. (supplied PDF, pp. 21–22). |
| Model families | Linear/logistic, trees/ensembles, SVM, KNN, Naive Bayes, clustering, neural nets, Transformers, bagging/boosting/stacking | This paper is supervised binary classification. Clustering is not a direct replacement for the supervised task; deep learning and additional models would expand scope. | Logistic Regression, Random Forest and XGBoost are the selected comparison models. SVM/KNN/NB/NN are possible future baselines, not default candidates. |
| Hyperparameter search | Default, small grid, GridSearchCV, RandomizedSearchCV, Bayesian/TPE/SMAC, successive halving, AutoML | Use a small explicit grid for a few finalists. If the grid becomes wide or continuous, RandomizedSearchCV or Bayesian optimization is more efficient. | GridSearchCV is used for compact pipeline and model searches. Search results, fold means/std and runtime are exported. (supplied PDF, pp. 27–29). |
| Evaluation | Accuracy, precision, recall, F1, ROC/PR-AUC, confusion matrix, calibration, thresholds, error/slice analysis, robustness, confidence intervals | Attack false negatives and benign false alarms both matter. Accuracy alone is insufficient under imbalance. | Report per-class and aggregate metrics, confusion matrices, ROC/PR curves, CV mean/std, fit time and inference latency. |
| Explainability | SHAP, LIME, permutation importance, PDP, counterfactuals, surrogate models, model cards/fairness | SHAP can explain model-dependent feature contributions and help investigate shortcuts; it does not establish causality. | SHAP global/local analysis for the selected tree model; optional permutation importance for comparison. LIME/counterfactuals not required for this paper. |
| Production lifecycle | Serialization, Docker, REST/gRPC, batch/stream/edge/cloud deployment, A/B/canary/shadow, monitoring, drift, retraining, CI/CD, registries, governance | Valid lifecycle topics but not necessary to answer the offline experimental research questions. | Out of scope; retain for future work, not this notebook's search space. |

### Model-dependency rules[¶](#Model-dependency-rules)

- Scaling is tested for Logistic Regression; it is not a useful generic search axis for Random Forest/XGBoost.
- Protocol is nominal, so one-hot encoding is the defensible primary representation; target encoding is unnecessary and brings fold-leakage risk.
- Samplers are inside an imbalanced-learn pipeline so they run only during each training fold.
- Feature selectors are inside the CV pipeline so the selector is refit within each training fold.
- Split choice is a design decision, not a score-optimization parameter.
- If the model family, metric, or data structure changes, the search space must be reconsidered.

The supplied framework explicitly identifies these dependencies and warns against comparing pipelines that change many factors at once. (supplied PDF, pp. 22–23 and 29–30).

## 2B. Search strategy and experiment ledger[¶](#2B.-Search-strategy-and-experiment-ledger)

The notebook stores the candidate method lists explicitly. The lists are not all multiplied into one enormous grid. Each eligible family is searched separately, then the best 2–4 defensible pipeline candidates are combined and validated. This is the staged factorial design recommended in the supplied framework.

In [1]:
METHOD_CATALOG = [
    {"family": "numeric_imputation", "method": "median", "status": "SEARCHED", "reason": "Robust baseline for skewed flow statistics."},
    {"family": "numeric_imputation", "method": "mean", "status": "SEARCHED", "reason": "Comparison baseline for less-skewed features."},
    {"family": "numeric_imputation", "method": "constant_zero", "status": "SEARCHED", "reason": "Simple sentinel baseline; interpret cautiously."},
    {"family": "numeric_imputation", "method": "KNN", "status": "DEFERRED", "reason": "Potentially expensive at CIC-IDS2017 scale."},
    {"family": "numeric_imputation", "method": "Iterative", "status": "DEFERRED", "reason": "More computationally expensive; only if simple methods are inadequate."},
    {"family": "numeric_imputation", "method": "missing_indicator", "status": "SEARCHED", "reason": "Tests whether missingness itself carries signal."},
    {"family": "categorical_imputation", "method": "most_frequent", "status": "CONDITIONAL", "reason": "Only if Protocol values are missing."},
    {"family": "categorical_imputation", "method": "constant_Missing", "status": "CONDITIONAL", "reason": "Only if Protocol values are missing."},
    {"family": "outlier_treatment", "method": "none", "status": "SEARCHED", "reason": "Preserve rare but potentially genuine attack flows."},
    {"family": "outlier_treatment", "method": "quantile_clip_1_99", "status": "SEARCHED", "reason": "Train-fitted clipping candidate."},
    {"family": "outlier_treatment", "method": "row_removal", "status": "NOT_DEFAULT", "reason": "Could delete rare attack events; only remove proven corrupt records."},
    {"family": "numeric_transform", "method": "Yeo-Johnson", "status": "SEARCHED_FOR_LR", "reason": "Handles negative and non-negative skewed values."},
    {"family": "numeric_transform", "method": "log1p", "status": "CONDITIONAL", "reason": "Only for individually verified non-negative, skewed variables."},
    {"family": "encoding", "method": "OneHotEncoder", "status": "SEARCHED", "reason": "Suitable for low-cardinality nominal Protocol."},
    {"family": "encoding", "method": "drop_Protocol", "status": "ABLATION", "reason": "Measures whether Protocol contributes predictive signal."},
    {"family": "encoding", "method": "ordinal_or_label", "status": "NOT_APPLICABLE", "reason": "Would create artificial order between protocol codes."},
    {"family": "encoding", "method": "target_frequency_hash_embedding", "status": "NOT_APPLICABLE", "reason": "Designed for high-cardinality categories; Protocol is low-cardinality."},
    {"family": "scaling", "method": "none", "status": "SEARCHED", "reason": "Tree model default; also a Logistic Regression candidate."},
    {"family": "scaling", "method": "standard", "status": "SEARCHED_FOR_LR", "reason": "Mean-zero/unit-variance scaling for Logistic Regression."},
    {"family": "scaling", "method": "robust", "status": "SEARCHED_FOR_LR", "reason": "Median/IQR scaling when features contain extreme values."},
    {"family": "scaling", "method": "minmax", "status": "SEARCHED_FOR_LR", "reason": "Bounded range candidate; sensitive to extreme values."},
    {"family": "feature_selection", "method": "none", "status": "SEARCHED", "reason": "Baseline with all eligible features."},
    {"family": "feature_selection", "method": "VarianceThreshold", "status": "SEARCHED", "reason": "Fold-local removal of zero-variance features."},
    {"family": "feature_selection", "method": "ANOVA_SelectKBest", "status": "SEARCHED", "reason": "Fast supervised filter for classification."},
    {"family": "feature_selection", "method": "MutualInformation_SelectKBest", "status": "OPTIONAL_SEARCH", "reason": "Nonlinear univariate dependence; enable after pilot."},
    {"family": "feature_selection", "method": "RFE_L1_tree_embedded_permutation", "status": "DEFERRED", "reason": "Additional compute and methodological complexity."},
    {"family": "feature_selection", "method": "SHAP_top_k", "status": "POST_FIT_EXPERIMENT", "reason": "Separate explainability-guided feature reduction."},
    {"family": "dimensionality_reduction", "method": "PCA", "status": "DEFERRED", "reason": "May reduce original-feature interpretability."},
    {"family": "dimensionality_reduction", "method": "tSNE_UMAP", "status": "VISUALIZATION_ONLY", "reason": "Not primary predictive features in this study."},
    {"family": "split", "method": "stratified_random", "status": "PRIMARY_DEFAULT", "reason": "Row-level IID benchmark; interpret duplicate-overlap audit."},
    {"family": "split", "method": "duplicate_grouped", "status": "ROBUSTNESS_OPTION", "reason": "Keep identical feature vectors in one partition."},
    {"family": "split", "method": "chronological", "status": "ROBUSTNESS_OPTION", "reason": "Later-time evaluation; attack-day scenario shift must be acknowledged."},
    {"family": "cross_validation", "method": "StratifiedKFold", "status": "PRIMARY_DEFAULT", "reason": "Maintains class proportions in row-level CV."},
    {"family": "cross_validation", "method": "StratifiedGroupKFold", "status": "CONDITIONAL", "reason": "Use with duplicate-grouped split."},
    {"family": "cross_validation", "method": "TimeSeriesSplit", "status": "CONDITIONAL", "reason": "Use with chronological split; some folds may lack one class."},
    {"family": "imbalance", "method": "none", "status": "SEARCHED", "reason": "Natural-prevalence baseline."},
    {"family": "imbalance", "method": "class_weight_or_scale_pos_weight", "status": "SEARCHED", "reason": "Cost-sensitive learning without synthetic rows."},
    {"family": "imbalance", "method": "RandomOverSampler", "status": "SEARCHED", "reason": "Duplicates minority training rows inside each fold."},
    {"family": "imbalance", "method": "RandomUnderSampler", "status": "SEARCHED", "reason": "Reduces majority training rows inside each fold."},
    {"family": "imbalance", "method": "SMOTE_SMOTENC_ADASYN_Tomek", "status": "DEFERRED", "reason": "Needs a categorical-aware, fold-local design and separate compute budget."},
    {"family": "model", "method": "LogisticRegression", "status": "SEARCHED", "reason": "Linear baseline."},
    {"family": "model", "method": "RandomForest", "status": "SEARCHED", "reason": "Bagging tree ensemble."},
    {"family": "model", "method": "XGBoost", "status": "SEARCHED_IF_INSTALLED", "reason": "Gradient-boosted tree ensemble."},
    {"family": "model", "method": "SVM_KNN_NaiveBayes_NN", "status": "FUTURE_WORK", "reason": "Would broaden model scope beyond the present paper."},
    {"family": "hyperparameter_search", "method": "GridSearchCV", "status": "SEARCHED", "reason": "Compact explicit discrete grids for shortlisted candidates."},
    {"family": "hyperparameter_search", "method": "Randomized_Bayesian_Halving_AutoML", "status": "FUTURE_OPTION", "reason": "More appropriate than a huge exhaustive grid if search spaces expand."},
]

## 3. UV environment[¶](#3.-UV-environment)

Run in PowerShell from the project root, not in a notebook `%pip` cell:

```
uv add numpy pandas scipy scikit-learn matplotlib seaborn xgboost shap pyarrow psutil imbalanced-learn
uv add --dev jupyterlab ipykernel
uv sync
uv run python -m ipykernel install --user --name cicids2017 --display-name "Python (CIC-IDS2017 - UV)"
uv run jupyter lab
```

Select **Python (CIC-IDS2017 - UV)** as the notebook kernel. `GridSearchCV` is provided by scikit-learn; `ImbPipeline` and the samplers are provided by imbalanced-learn. We are not adding MLflow, Docker, CI/CD or pipeline orchestration to this experimental notebook.

## 4. Imports and environment verification[¶](#4.-Imports-and-environment-verification)

In [2]:
from __future__ import annotations

import json, math, os, platform, re, time, warnings, hashlib
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

import sklearn
import joblib
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, PowerTransformer
)
from sklearn.feature_selection import (
    SelectKBest, VarianceThreshold, f_classif, mutual_info_classif
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit,
    GridSearchCV, RandomizedSearchCV, cross_validate
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score, balanced_accuracy_score,
    roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", repr(exc))

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    SHAP_AVAILABLE = False
    print("SHAP unavailable:", repr(exc))

try:
    import psutil
except Exception:
    psutil = None

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("XGBoost:", XGBOOST_AVAILABLE, "| SHAP:", SHAP_AVAILABLE)

Python: 3.11.16
pandas: 3.1.1
NumPy: 2.4.6
scikit-learn: 1.9.1
XGBoost: True | SHAP: True


e:\03 Education\Zagazig Material\Masters\First Year First Term\Machine Learning\Research Paper\ML for Intrusion Detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5. Project paths and experiment settings[¶](#5.-Project-paths-and-experiment-settings)

In [3]:
def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the project or set PROJECT_DIR manually.")

PROJECT_DIR = find_project_root()
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_RUN_NAME = "cicids2017_research_v3_gpu_external"
OUTPUT_ROOT = PROJECT_DIR / "research_outputs_v3"
AUDIT_DIR = OUTPUT_ROOT / "audit"
FIGURES_DIR = OUTPUT_ROOT / "figures"
TABLES_DIR = OUTPUT_ROOT / "tables"

for folder in [AUDIT_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
CHUNK_SIZE = 100_000
MAX_ROWS_FOR_MODELING = None    
SEARCH_MAX_ROWS = 25_000           # Separate training-only CV subset; does NOT cap the initial data load
TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15
CV_FOLDS = 3
SEARCH_N_JOBS = 1                  # avoid nested CPU oversubscription during GridSearchCV
MODEL_N_JOBS = -1

# GPU policy: XGBoost is required to use CUDA; no silent CPU fallback.
XGB_DEVICE = "cuda"
XGB_REQUIRE_GPU = True
XGB_N_JOBS = 4
PRIMARY_SCORING = "f1_macro"

# AUC scorers can be undefined for a time-based fold containing only one class.
SCORING = {
    "f1_macro": "f1_macro",
    "f1_attack": "f1",
    "precision_attack": "precision",
    "recall_attack": "recall",
    "balanced_accuracy": "balanced_accuracy",
}

# Add threshold-independent ranking metrics for stratified/grouped folds.
# For chronological CV, these can be undefined when a fold has one class.
SPLIT_STRATEGY = "stratified_random"  # stratified_random | duplicate_grouped | chronological
CV_STRATEGY = {
    "stratified_random": "stratified",
    "duplicate_grouped": "duplicate_grouped",
    "chronological": "chronological",
}[SPLIT_STRATEGY]
if CV_STRATEGY != "chronological":
    SCORING.update({"roc_auc": "roc_auc", "average_precision": "average_precision"})

REMOVE_EXACT_DUPLICATES = False   # default: audit but retain; compare removal only as a separate study
SHAP_SAMPLE_SIZE = 750
TOP_K_FEATURES = [10, 20, 30]
INCLUDE_MUTUAL_INFO_IN_SEARCH = True

# Search budget. The CV subset is only for choosing configurations; final refits use train+validation.
BASE_RF_ESTIMATORS = 100
BASE_XGB_ESTIMATORS = 80
XGB_SEARCH_N_ESTIMATORS = [100, 180]
RF_SEARCH_N_ESTIMATORS = [100, 200]

IDENTIFIER_COLUMNS = {
    "flow id", "source ip", "destination ip", "timestamp", "source_file", "row_id"
}
LABEL_NAMES = {"label", "label_original", "label_normalized", "target"}

print("Project:", PROJECT_DIR)
print("Raw:", RAW_DIR)
print("Split:", SPLIT_STRATEGY, "| CV:", CV_STRATEGY)
print("Modeling sample cap:", MAX_ROWS_FOR_MODELING, "| CV search subset cap:", SEARCH_MAX_ROWS)
print("CV folds:", CV_FOLDS)
print("Output namespace:", OUTPUT_ROOT)
print("XGBoost requested device:", XGB_DEVICE)


Project: E:\03 Education\Zagazig Material\Masters\First Year First Term\Machine Learning\Research Paper\ML for Intrusion Detection
Raw: E:\03 Education\Zagazig Material\Masters\First Year First Term\Machine Learning\Research Paper\ML for Intrusion Detection\data\raw
Split: stratified_random | CV: stratified
Modeling sample cap: None | CV search subset cap: 25000
CV folds: 3
Output namespace: E:\03 Education\Zagazig Material\Masters\First Year First Term\Machine Learning\Research Paper\ML for Intrusion Detection\research_outputs_v3
XGBoost requested device: cuda


## 6. Reusable result and shape helpers[¶](#6.-Reusable-result-and-shape-helpers)

In [ ]:
def save_table(frame, filename):
    path = TABLES_DIR / filename
    frame.to_csv(path, index=False)
    return path

def record_shape(stage, frame, target=None):
    row = {
        "stage": stage,
        "rows": int(frame.shape[0]),
        "columns": int(frame.shape[1]),
        "memory_mb": float(frame.memory_usage(deep=True).sum() / 1024**2),
    }
    if target is not None:
        counts = target.value_counts().to_dict()
        row["benign"] = int(counts.get(0, 0))
        row["attack"] = int(counts.get(1, 0))
    return row

def calculate_metrics(y_true, y_pred, y_probability):
    """Comprehensive binary metrics; class 1 is Attack and class 0 is Benign."""
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.int8)
    y_probability = np.asarray(y_probability, dtype=float)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    both_classes = np.unique(y_true).size == 2
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    fnr = fn / (fn + tp) if (fn + tp) else np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "error_rate": 1 - accuracy_score(y_true, y_pred),
        "precision_attack": precision,
        "recall_attack": recall,
        "specificity_benign": specificity,
        "npv_benign": tn / (tn + fn) if (tn + fn) else np.nan,
        "f1_attack": f1_score(y_true, y_pred, zero_division=0),
        "f2_attack": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "jaccard_attack": tp / (tp + fp + fn) if (tp + fp + fn) else np.nan,
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_probability) if both_classes else np.nan,
        "average_precision": average_precision_score(y_true, y_probability) if both_classes else np.nan,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": sklearn.metrics.cohen_kappa_score(y_true, y_pred),
        "g_mean": math.sqrt(max(0.0, recall * specificity)) if np.isfinite(specificity) else np.nan,
        "false_positive_rate": fpr,
        "false_negative_rate": fnr,
        "false_discovery_rate": fp / (fp + tp) if (fp + tp) else np.nan,
        "false_omission_rate": fn / (fn + tn) if (fn + tn) else np.nan,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "support_benign": int(tn + fp), "support_attack": int(fn + tp),
    }

def rank_validation_results(frame):
    # Primary rank: Macro-F1. Tie-breakers: Attack Recall, then lower FPR.
    return frame.sort_values(
        ["f1_macro", "recall_attack", "false_positive_rate"],
        ascending=[False, False, True], na_position="last"
    ).reset_index(drop=True)

def stratified_search_subset(X, y, max_rows, seed=RANDOM_STATE):
    if max_rows is None or len(X) <= max_rows:
        return X.copy(), y.copy()
    if max_rows < 4:
        raise ValueError("SEARCH_MAX_ROWS is too small.")
    X_sub, _, y_sub, _ = train_test_split(
        X, y, train_size=max_rows, random_state=seed, stratify=y
    )
    return X_sub.copy(), y_sub.copy()

def show_table(frame, title=None, max_rows=30, columns=None, precision=4, show_index=False):
    """Render compact, readable, horizontally scrollable notebook tables."""
    if frame is None:
        print(f"{title or 'Table'}: no data")
        return
    view = frame.copy()
    if columns is not None:
        available = [c for c in columns if c in view.columns]
        missing = [c for c in columns if c not in view.columns]
        view = view.loc[:, available]
        if missing:
            print("Unavailable display columns:", missing)
    total_rows = len(view)
    if max_rows is not None:
        view = view.head(max_rows)

    def _fmt(value):
        if value is None or value is pd.NA:
            return "—"
        if isinstance(value, (float, np.floating)):
            if np.isnan(value):
                return "—"
            if np.isinf(value):
                return "∞" if value > 0 else "−∞"
            return f"{value:,.{precision}f}"
        if isinstance(value, (int, np.integer)) and not isinstance(value, (bool, np.bool_)):
            return f"{value:,}"
        return str(value)

    formatters = {column: _fmt for column in view.columns}
    table_html = view.to_html(
        index=show_index, border=0, escape=True, formatters=formatters,
        classes="cicids-research-table", justify="left"
    )
    heading = f"<h4>{title}</h4>" if title else ""
    truncation = (
        f"<div class='cicids-table-note'>Showing {len(view):,} of {total_rows:,} rows.</div>"
        if total_rows > len(view) else ""
    )
    style = """
    <style>
      .cicids-table-wrap{max-width:100%;overflow-x:auto;margin:8px 0 18px 0;}
      table.cicids-research-table{border-collapse:collapse;font-size:12.5px;white-space:nowrap;}
      table.cicids-research-table th{background:#263445;color:#fff;padding:7px 10px;text-align:left;position:sticky;top:0;}
      table.cicids-research-table td{border-bottom:1px solid #d9dee7;padding:6px 10px;}
      table.cicids-research-table tr:nth-child(even){background:#f4f6f9;}
      .cicids-table-note{font-size:11px;color:#687386;margin-top:5px;}
    </style>
    """
    display(HTML(style + f"<div class='cicids-table-wrap'>{heading}{table_html}{truncation}</div>"))

In [ ]:
method_catalog_df = pd.DataFrame(METHOD_CATALOG)
show_table(method_catalog_df, "Method catalogue and search disposition", max_rows=60)
save_table(method_catalog_df, "method_catalog_and_search_disposition.csv")

## 7. Raw file inventory and schema validation[¶](#7.-Raw-file-inventory-and-schema-validation)

The ZIP is inspected by filename only; the notebook will not extract it automatically. The eight extracted CSV files are the modeling input.

In [ ]:
if not RAW_DIR.exists():
    raise FileNotFoundError(f"Raw-data folder not found: {RAW_DIR}")
csv_files = sorted(RAW_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found under {RAW_DIR}")

inventory_df = pd.DataFrame([
    {"file": p.name, "size_mb": round(p.stat().st_size / 1024**2, 2),
     "size_bytes": p.stat().st_size}
    for p in csv_files
])
show_table(inventory_df, "Raw CSV inventory")
print("CSV count:", len(csv_files), "| ZIP files:", [p.name for p in RAW_DIR.glob("*.zip")])
save_table(inventory_df, "raw_file_inventory.csv")

def read_csv_header(path):
    last_error = None
    for encoding in ["utf-8-sig", "cp1252", "latin1"]:
        try:
            header = pd.read_csv(path, nrows=0, encoding=encoding)
            columns = [str(c).strip() for c in header.columns]
            if len(columns) != len(set(columns)):
                raise ValueError(f"Duplicate headers after strip: {path.name}")
            return columns, encoding
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Could not read {path.name}: {last_error}")

header_info = {}
for path in csv_files:
    columns, encoding = read_csv_header(path)
    header_info[path.name] = {"columns": columns, "encoding": encoding}

reference_columns = header_info[csv_files[0].name]["columns"]
schema_rows = []
for path in csv_files:
    cols = header_info[path.name]["columns"]
    schema_rows.append({
        "file": path.name,
        "encoding": header_info[path.name]["encoding"],
        "n_columns": len(cols),
        "matches_reference": cols == reference_columns,
        "missing_columns": sorted(set(reference_columns) - set(cols)),
        "extra_columns": sorted(set(cols) - set(reference_columns)),
    })

schema_audit_df = pd.DataFrame(schema_rows)
show_table(schema_audit_df, "CSV schema consistency audit")
save_table(schema_audit_df, "schema_audit.csv")
if not schema_audit_df["matches_reference"].all():
    raise ValueError("CSV schemas differ; resolve before concatenating.")

label_matches = [c for c in reference_columns if c.strip().lower() == "label"]
if len(label_matches) != 1:
    raise ValueError(f"Expected one Label column, found: {label_matches}")
LABEL_COLUMN = label_matches[0]

## 8. Full-dataset label audit using chunks[¶](#8.-Full-dataset-label-audit-using-chunks)

In [ ]:
def audit_labels_in_file(path, encoding, chunksize=CHUNK_SIZE):
    """Count raw labels in chunks while reading only the Label column.

    LABEL_COLUMN is normalized during header inspection, but pandas usecols
    matches the original CSV header spelling. The callable below handles
    headers such as " Label" and "Label ".
    """
    counts = Counter()
    total_rows = 0

    def is_label_column(raw_name):
        normalized = str(raw_name).replace("\ufeff", "").strip().casefold()
        return normalized == "label"

    for chunk in pd.read_csv(
        path,
        usecols=is_label_column,
        chunksize=chunksize,
        encoding=encoding,
        low_memory=False,
    ):
        if chunk.shape[1] != 1:
            raise ValueError(
                f"Expected exactly one Label column in {path.name}; "
                f"read {chunk.shape[1]} columns: {chunk.columns.tolist()}"
            )

        # usecols keeps the raw spelling; iloc avoids relying on its exact name.
        labels = chunk.iloc[:, 0].astype("string").str.strip()
        labels = labels.mask(labels.eq(""), pd.NA)

        for label, count in labels.value_counts(dropna=False).items():
            key = "<MISSING>" if pd.isna(label) else str(label)
            counts[key] += int(count)

        total_rows += len(chunk)

    return total_rows, counts

audit_rows = []
for path in csv_files:
    n_rows, counts = audit_labels_in_file(path, header_info[path.name]["encoding"])
    print(f"\n{path.name}: {n_rows:,} rows")
    for label, count in counts.most_common():
        print(f"  {label}: {count:,}")
        audit_rows.append({"file": path.name, "label": label, "count": count, "file_rows": n_rows})

label_audit_df = pd.DataFrame(audit_rows)
show_table(label_audit_df, "Original label counts by CSV", max_rows=100)
save_table(label_audit_df, "label_distribution_by_file.csv")
full_label_counts = label_audit_df.groupby("label")["count"].sum().sort_values(ascending=False)
print("Total rows in all CSVs:", int(label_audit_df.drop_duplicates("file")["file_rows"].sum()))
show_table(full_label_counts.rename_axis("original_label").to_frame("count"), "Full-dataset label distribution", max_rows=30)

## 9. Load data: full mode or reproducible stratified streaming sample[¶](#9.-Load-data:-full-mode-or-reproducible-stratified-streaming-sample)

**Important distinction:** `MAX_ROWS_FOR_MODELING` controls how many rows are loaded from the eight source CSVs. `SEARCH_MAX_ROWS` is applied later, after the train/validation/test split, and limits only the training subset used for cross-validation searches. It does not limit the initial load or validation/final-fit rows. With the reported 2.6 GB available RAM, the default modeling sample is set to 200,000 rows. The full label audit above still scans every source row in chunks. EDA and duplicate diagnostics below describe the modeling sample unless a cell explicitly says otherwise.

The loader adds a `source_file` provenance column to each row. Each original CSV has 79 columns; the loaded frame has 80 because this additional metadata column is deliberately retained for auditing. It is excluded from `FEATURE_COLUMNS` and never used by classifiers.

The reservoir sampler keeps the smallest uniform random keys per original label across all chunks. It retains each row's random key until the sampling pass finishes, so the reservoir remains valid across chunks.

In [ ]:
def allocate_label_quotas(label_counts, total_n):
    counts = label_counts[label_counts.index != "<MISSING>"].astype(int)
    if total_n <= 0 or total_n > int(counts.sum()):
        raise ValueError("Sample size must be between 1 and the number of labeled rows.")
    raw = counts / counts.sum() * total_n
    quotas = np.floor(raw).astype(int)
    remainder = total_n - int(quotas.sum())
    for label in (raw - quotas).sort_values(ascending=False).index[:remainder]:
        quotas.loc[label] += 1
    for label in counts[counts > 0].index:
        if quotas.loc[label] == 0 and total_n >= len(counts):
            donor = quotas.idxmax()
            quotas.loc[donor] -= 1
            quotas.loc[label] = 1
    return quotas.to_dict()

def load_stratified_sample_from_chunks(paths, quotas, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    reservoirs = {label: pd.DataFrame() for label in quotas}
    for path in paths:
        encoding = header_info[path.name]["encoding"]
        for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE, encoding=encoding, low_memory=False):
            chunk.columns = chunk.columns.astype(str).str.strip()
            labels = chunk[LABEL_COLUMN].astype("string").str.strip()
            labels = labels.mask(labels.eq(""), pd.NA)
            valid = labels.notna()
            chunk = chunk.loc[valid].copy()
            chunk["_sampling_label"] = labels.loc[valid].astype(str).to_numpy()
            chunk["source_file"] = path.name
            chunk["_random_key"] = rng.random(len(chunk))
            for label, quota in quotas.items():
                if quota <= 0:
                    continue
                current = chunk.loc[chunk["_sampling_label"] == label]
                if not current.empty:
                    reservoirs[label] = pd.concat(
                        [reservoirs[label], current], ignore_index=True
                    ).nsmallest(quota, "_random_key")
    sampled = pd.concat(
        [frame for frame in reservoirs.values() if not frame.empty], ignore_index=True
    )
    sampled = sampled.drop(columns=["_sampling_label", "_random_key"])
    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)

if MAX_ROWS_FOR_MODELING is None:
    if psutil is not None:
        available_gb = psutil.virtual_memory().available / 1024**3
        csv_size_gb = sum(p.stat().st_size for p in csv_files) / 1024**3
        print(f"Available RAM: {available_gb:.1f} GB | CSV size: {csv_size_gb:.2f} GB")
        if available_gb < 3 * csv_size_gb:
            warnings.warn("RAM may be insufficient for full pandas loading; consider a documented sample.")
    parts = []
    for path in csv_files:
        print("Loading:", path.name)
        part = pd.read_csv(path, encoding=header_info[path.name]["encoding"], low_memory=False)
        part.columns = part.columns.astype(str).str.strip()
        part["source_file"] = path.name
        parts.append(part)
    data = pd.concat(parts, ignore_index=True, sort=False)
    del parts
    sampling_description = "All rows from all supplied CSV files."
else:
    quotas = allocate_label_quotas(full_label_counts, int(MAX_ROWS_FOR_MODELING))
    print("Per-label quotas:", quotas)
    data = load_stratified_sample_from_chunks(csv_files, quotas)
    sampling_description = f"Stratified streaming sample capped at {MAX_ROWS_FOR_MODELING:,} rows."

print("Loaded shape:", data.shape)
print("Original source-schema columns:", len(reference_columns))
print("Added provenance column:", "source_file" if "source_file" in data.columns else "NOT FOUND")
print("DataFrame memory GB:", round(data.memory_usage(deep=True).sum()/1024**3, 2))
print("Sampling:", sampling_description)
if MAX_ROWS_FOR_MODELING is not None and len(data) > MAX_ROWS_FOR_MODELING:
    raise RuntimeError("Loaded rows exceed MAX_ROWS_FOR_MODELING.")
preview_columns = [c for c in ["Destination Port", "Flow Duration", "Total Fwd Packets", "Flow Bytes/s", "Flow Packets/s", LABEL_COLUMN, "source_file"] if c in data.columns]
show_table(data.loc[:, preview_columns].head(10), "Sampled dataset preview (selected columns)", max_rows=10)

## 10. Binary target and original label distribution[¶](#10.-Binary-target-and-original-label-distribution)

Keep the original attack family label for descriptive statistics. The target is `BENIGN = 0`, all other non-empty original labels = `ATTACK = 1`. The binary task does not claim attack-family-level performance.

In [ ]:
label_matches = [c for c in data.columns if c.strip().lower() == "label"]
if len(label_matches) != 1:
    raise ValueError(f"Expected one Label field, found {label_matches}")
LABEL_COLUMN = label_matches[0]

data["label_original"] = data[LABEL_COLUMN].astype("string").str.strip()
data["label_original"] = data["label_original"].mask(data["label_original"].eq(""), pd.NA)
missing_label_count = int(data["label_original"].isna().sum())
data = data.loc[data["label_original"].notna()].copy()
data["label_normalized"] = data["label_original"].str.upper()
data["target"] = (data["label_normalized"] != "BENIGN").astype("int8")

original_label_counts = (
    data["label_original"].value_counts().rename_axis("original_label")
    .reset_index(name="count")
)
binary_counts = data["target"].value_counts().reindex([0, 1], fill_value=0)

print("Missing-label rows excluded:", missing_label_count)
show_table(original_label_counts, "Original attack-family distribution in modeling sample", max_rows=30)
show_table(binary_counts.rename(index={0: "Benign (0)", 1: "Attack (1)"}).rename_axis("target_class").to_frame("count"), "Binary target distribution")
save_table(original_label_counts, "original_label_counts.csv")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(original_label_counts["original_label"], original_label_counts["count"])
axes[0].set_title("Original CIC-IDS2017 labels")
axes[0].tick_params(axis="x", rotation=80)
axes[1].bar(["Benign", "Attack"], [binary_counts.loc[0], binary_counts.loc[1]])
axes[1].set_title("Binary target distribution")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "label_distributions.png", dpi=180)
plt.show()

## 11. Feature typing and deterministic conversion[¶](#11.-Feature-typing-and-deterministic-conversion)

`Flow ID`, IP addresses, timestamp, and the notebook-added `source_file` column are metadata rather than predictor columns. `source_file` explains why the loaded frame has 80 columns although each original CSV has 79: it is intentionally added during ingestion to preserve row provenance. `Protocol` is categorical even though CICFlowMeter records it as a number. Source/destination ports are kept initially but scrutinized as potential dataset-specific shortcuts.

In [ ]:
# CIC-IDS2017 commonly exports a duplicated "Fwd Header Length" column as
# "Fwd Header Length.1". Remove it ONLY after verifying row-wise equality.
DUPLICATE_FEATURE_AUDIT = []
if "Fwd Header Length" in data.columns and "Fwd Header Length.1" in data.columns:
    _left = pd.to_numeric(data["Fwd Header Length"], errors="coerce").to_numpy(dtype=float)
    _right = pd.to_numeric(data["Fwd Header Length.1"], errors="coerce").to_numpy(dtype=float)
    _same = np.allclose(_left, _right, rtol=0.0, atol=0.0, equal_nan=True)
    DUPLICATE_FEATURE_AUDIT.append({
        "feature_a": "Fwd Header Length", "feature_b": "Fwd Header Length.1",
        "identical_values": bool(_same), "rows_checked": int(len(data)),
        "action": "drop duplicated .1 column" if _same else "retain both; values differ"
    })
    if _same:
        data = data.drop(columns=["Fwd Header Length.1"])
    else:
        warnings.warn("Fwd Header Length.1 is not identical; it will remain a separate source feature.")
if DUPLICATE_FEATURE_AUDIT:
    duplicate_feature_audit_df = pd.DataFrame(DUPLICATE_FEATURE_AUDIT)
    show_table(duplicate_feature_audit_df, "Verified duplicate-feature audit")
    save_table(duplicate_feature_audit_df, "verified_duplicate_feature_audit.csv")

drop_lower = IDENTIFIER_COLUMNS | LABEL_NAMES
drop_columns = [
    c for c in data.columns
    if c.strip().lower() in drop_lower or c.strip().lower().startswith("unnamed:")
]
candidate_columns = [c for c in data.columns if c not in drop_columns]
protocol_matches = [c for c in candidate_columns if c.strip().lower() == "protocol"]
if len(protocol_matches) > 1:
    raise ValueError(f"Multiple Protocol columns: {protocol_matches}")
PROTOCOL_COLUMN = protocol_matches[0] if protocol_matches else None

NUMERIC_FEATURES = [c for c in candidate_columns if c != PROTOCOL_COLUMN]
conversion_rows = []
for col in NUMERIC_FEATURES:
    raw = data[col].astype("string").str.strip().replace("", pd.NA)
    original_missing = int(raw.isna().sum())
    nonmissing_before = int(raw.notna().sum())
    converted = pd.to_numeric(raw, errors="coerce")
    parse_failures = nonmissing_before - int(converted.notna().sum())
    positive_infinity = int(np.isposinf(converted.to_numpy(dtype=float, na_value=np.nan)).sum())
    negative_infinity = int(np.isneginf(converted.to_numpy(dtype=float, na_value=np.nan)).sum())
    cleaned = converted.replace([np.inf, -np.inf], np.nan)
    conversion_rows.append({
        "feature": col,
        "original_missing_or_blank": original_missing,
        "nonmissing_before_conversion": nonmissing_before,
        "parse_failures": parse_failures,
        "positive_infinity": positive_infinity,
        "negative_infinity": negative_infinity,
        "missing_after_numeric_cleaning": int(cleaned.isna().sum()),
    })
    data[col] = cleaned.astype("float32")

CATEGORICAL_FEATURES = [PROTOCOL_COLUMN] if PROTOCOL_COLUMN else []
if PROTOCOL_COLUMN:
    p = data[PROTOCOL_COLUMN].astype("string").str.strip()
    p = p.mask(p.eq(""), pd.NA)
    # Convert pandas StringDtype missing markers to NumPy NaN for sklearn SimpleImputer.
    data[PROTOCOL_COLUMN] = pd.Series(
        p.to_numpy(dtype=object, na_value=np.nan), index=data.index, dtype=object
    )

conversion_audit_df = pd.DataFrame(conversion_rows)

conversion_audit_df = conversion_audit_df.sort_values(
    by=["parse_failures", "missing_after_numeric_cleaning"],
    ascending=False
)

show_table(
    conversion_audit_df.head(30),
    "Numeric conversion and invalid-value audit",
    max_rows=30
)

save_table(conversion_audit_df, "numeric_conversion_audit.csv")

if (conversion_audit_df["parse_failures"] > 0).any():
    warnings.warn(
        "Some non-missing predictor values could not be converted to numeric. "
        "Inspect the conversion audit."
    )

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
print("Excluded metadata/target:", drop_columns)
print("Numeric feature count:", len(NUMERIC_FEATURES))
print("Categorical features:", CATEGORICAL_FEATURES)
print("Total features:", len(FEATURE_COLUMNS))

## 12. EDA and data-quality audit[¶](#12.-EDA-and-data-quality-audit)

We report missingness, numeric ranges, skewness, constant features, duplicates, and conflicting labels for identical feature vectors. IQR outlier flags and negative-value counts are diagnostic only; they do not delete rows. The duplicate-group summary checks a logical invariant: a binary-label conflict must also be an original-label conflict.
**Scope note:** with `MAX_ROWS_FOR_MODELING=200_000`, these row-level EDA and duplicate statistics describe the reproducible modeling sample, not every one of the 2,830,743 raw records. The label distribution in Section 8 is computed over all raw CSV rows using chunked reading. We retain this full-corpus versus sample distinction in the paper; do not report sample-only duplicate counts as full-dataset counts.

In [ ]:
missing_summary = data[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False).to_frame("missing_count")
missing_summary["missing_percent"] = 100 * missing_summary["missing_count"] / max(1, len(data))
show_table(missing_summary.head(40).reset_index(names="feature"), "Feature missingness", max_rows=40)
save_table(missing_summary.reset_index(names="feature"), "feature_missingness.csv")

numeric_summary = data[NUMERIC_FEATURES].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T
numeric_summary["skewness"] = data[NUMERIC_FEATURES].skew(numeric_only=True)
show_table(numeric_summary.head(40).reset_index(names="feature"), "Numerical feature summary", max_rows=40)
save_table(numeric_summary.reset_index(names="feature"), "numeric_feature_summary.csv")

constant_features = [c for c in FEATURE_COLUMNS if data[c].nunique(dropna=True) <= 1]
exact_full_duplicates = int(data.duplicated().sum())
exact_feature_target_duplicates = int(data.duplicated(subset=FEATURE_COLUMNS + ["target"]).sum())

feature_hashes = pd.util.hash_pandas_object(data[FEATURE_COLUMNS], index=False).astype("uint64")
hash_label = pd.DataFrame({
    "feature_hash": feature_hashes.to_numpy(),
    "binary_target": data["target"].to_numpy(),
    "original_label": data["label_original"].astype(str).to_numpy(),
})
# Each hash group represents one candidate identical feature vector. A group
# with both binary targets must also contain more than one original label.
feature_group_audit = hash_label.groupby("feature_hash", sort=False).agg(
    row_count=("feature_hash", "size"),
    binary_label_count=("binary_target", "nunique"),
    original_label_count=("original_label", "nunique"),
)
conflicting_binary_groups = feature_group_audit[feature_group_audit["binary_label_count"] > 1]
conflicting_original_groups = feature_group_audit[feature_group_audit["original_label_count"] > 1]
conflicting_binary_vectors = int(len(conflicting_binary_groups))
conflicting_original_label_vectors = int(len(conflicting_original_groups))

if conflicting_binary_vectors > conflicting_original_label_vectors:
    raise AssertionError(
        "Duplicate audit inconsistency: binary-conflicting feature groups cannot exceed "
        "groups with multiple original labels. Recheck target construction and index alignment."
    )

duplicate_audit = pd.DataFrame([{
    "rows_in_modeling_sample": len(data),
    "unique_feature_hash_groups": int(len(feature_group_audit)),
    "exact_full_row_duplicate_rows_after_first": exact_full_duplicates,
    "duplicate_feature_and_binary_target_rows_after_first": exact_feature_target_duplicates,
    "feature_groups_with_conflicting_binary_targets": conflicting_binary_vectors,
    "feature_groups_with_multiple_original_labels": conflicting_original_label_vectors,
    "constant_features_in_modeling_sample": len(constant_features),
    "constant_feature_names": ", ".join(constant_features),
}])
show_table(duplicate_audit, "Duplicate and conflicting-label audit")
save_table(duplicate_audit, "duplicate_audit.csv")

negative_value_summary = pd.DataFrame({
    "feature": NUMERIC_FEATURES,
    "negative_count": [int(data[c].lt(0).sum()) for c in NUMERIC_FEATURES],
})
negative_value_summary["negative_percent"] = (
    100 * negative_value_summary["negative_count"] / max(1, len(data))
)
negative_value_summary = negative_value_summary.sort_values("negative_count", ascending=False)
show_table(negative_value_summary.head(30), "Negative numeric values — investigate, do not automatically delete", max_rows=30)
save_table(negative_value_summary, "negative_value_audit.csv")

if REMOVE_EXACT_DUPLICATES:
    before = len(data)
    data = data.drop_duplicates(subset=FEATURE_COLUMNS + ["target"]).copy()
    print("Removed exact feature+binary-target duplicate rows:", before - len(data))
else:
    print("No duplicates removed; duplicates remain visible for split/leakage analysis.")

# IQR values below are descriptive audit flags only. They are NOT used to fit a clipper.
outlier_rows = []
for col in NUMERIC_FEATURES:
    s = data[col].dropna()
    if s.empty:
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    flagged = int(((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum())
    outlier_rows.append({
        "feature": col, "iqr_flagged_count": flagged,
        "iqr_flagged_pct": 100 * flagged / len(s),
        "q01": s.quantile(0.01), "median": s.median(), "q99": s.quantile(0.99)
    })
outlier_audit_df = pd.DataFrame(outlier_rows).sort_values("iqr_flagged_pct", ascending=False)
show_table(outlier_audit_df.head(30), "IQR outlier flags (diagnostic only)", max_rows=30)
save_table(outlier_audit_df, "iqr_outlier_audit.csv")

fig, ax = plt.subplots(figsize=(10, 6))
missing_summary.head(20).sort_values("missing_percent")["missing_percent"].plot(kind="barh", ax=ax)
ax.set_title("Top 20 features by missingness")
ax.set_xlabel("Missing values (%)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "missingness_top20.png", dpi=180)
plt.show()

## 13. Split strategy: choose by the generalization question[¶](#13.-Split-strategy:-choose-by-the-generalization-question)

- **Stratified random:** conventional IID row-level benchmark.
- **Duplicate-grouped:** identical modeling feature vectors stay together; this does not remove shared five-tuples or temporal dependence.
- **Chronological:** train on earlier rows and evaluate on later rows. On CIC-IDS2017, attack schedules differ by day, so a temporal split is also a scenario-shift experiment.

These alternatives are not optimized by score. Choose one before training and interpret the result accordingly.

In [ ]:
def make_holdout_indices(X, y, strategy, timestamp=None, groups=None):
    indices = np.arange(len(X))
    if strategy == "stratified_random":
        train_val, test = train_test_split(
            indices, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
        )
        val_fraction = VALIDATION_SIZE / (1 - TEST_SIZE)
        train, val = train_test_split(
            train_val, test_size=val_fraction, random_state=RANDOM_STATE,
            stratify=np.asarray(y)[train_val]
        )
    elif strategy == "duplicate_grouped":
        if groups is None:
            raise ValueError("Duplicate groups are required.")
        splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        splits = list(splitter.split(X, y, groups))
        train_val, test = min(splits, key=lambda s: abs(len(s[1])/len(X) - TEST_SIZE))
        inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 1)
        inner_splits = list(inner.split(
            X.iloc[train_val], np.asarray(y)[train_val], np.asarray(groups)[train_val]
        ))
        tr_rel, val_rel = min(
            inner_splits,
            key=lambda s: abs(len(s[1])/len(train_val) - VALIDATION_SIZE/(1-TEST_SIZE))
        )
        train, val = np.asarray(train_val)[tr_rel], np.asarray(train_val)[val_rel]
    elif strategy == "chronological":
        if timestamp is None:
            raise ValueError("Timestamp column is required for chronological split.")
        parsed = pd.to_datetime(timestamp, errors="coerce")
        if parsed.isna().any():
            raise ValueError("Invalid timestamps: cannot perform chronological split.")
        order = np.argsort(parsed.to_numpy())
        n = len(order)
        train_end = int(n * (1 - TEST_SIZE - VALIDATION_SIZE))
        val_end = int(n * (1 - TEST_SIZE))
        train, val, test = order[:train_end], order[train_end:val_end], order[val_end:]
    else:
        raise ValueError(strategy)
    return np.asarray(train), np.asarray(val), np.asarray(test)

X_all = data[FEATURE_COLUMNS].copy()
y_all = data["target"].astype("int8").copy()
duplicate_groups = pd.util.hash_pandas_object(X_all, index=False).astype("uint64").to_numpy()
timestamp_col = next((c for c in data.columns if c.strip().lower() == "timestamp"), None)
timestamp_values = data[timestamp_col] if timestamp_col else None

if SPLIT_STRATEGY == "chronological" and MAX_ROWS_FOR_MODELING is not None:
    warnings.warn("Chronological evaluation should preferably use all rows or a time-aware sample, not a class-stratified streaming sample.")

train_idx, val_idx, test_idx = make_holdout_indices(
    X_all, y_all, SPLIT_STRATEGY, timestamp=timestamp_values, groups=duplicate_groups
)
X_train, y_train = X_all.iloc[train_idx].copy(), y_all.iloc[train_idx].copy()
X_val, y_val = X_all.iloc[val_idx].copy(), y_all.iloc[val_idx].copy()
X_test, y_test = X_all.iloc[test_idx].copy(), y_all.iloc[test_idx].copy()

GROUPS_TRAIN = duplicate_groups[train_idx]
GROUPS_VAL = duplicate_groups[val_idx]
GROUPS_TEST = duplicate_groups[test_idx]

# Do NOT remove constant/all-empty features using the entire train partition here:
# that would let CV validation folds influence a fitted preprocessing decision.
# VarianceThreshold is available as a fold-fitted selector inside the pipeline instead.
NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c in X_train.columns]
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in X_train.columns]
FEATURES = X_train.columns.tolist()

split_shapes = pd.DataFrame([
    record_shape("train", X_train, y_train),
    record_shape("validation", X_val, y_val),
    record_shape("test", X_test, y_test)
])
show_table(split_shapes, "Train / validation / test partition sizes")
save_table(split_shapes, "split_shapes.csv")

for name, y_part in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(name, y_part.value_counts(normalize=True).sort_index().round(4).to_dict())
    if y_part.nunique() < 2:
        warnings.warn(f"{name} has one class; ROC-AUC/PR-AUC is undefined.")

constant_train_features = [c for c in X_train.columns if X_train[c].nunique(dropna=True) <= 1]
print("Constant/all-missing predictors observed in training partition (not globally dropped):", constant_train_features)

## 14. Cross-partition duplicate diagnostic and CV subset[¶](#14.-Cross-partition-duplicate-diagnostic-and-CV-subset)

The hash diagnostic measures exact feature-vector overlap between train and test. It is not a complete leakage audit: it does not identify near-duplicates, shared network connections, or temporal dependencies.

For chronological CV, we preserve temporal row order and do not randomly subsample the training rows. For other strategies, `SEARCH_MAX_ROWS` may cap the training-only tuning subset.

In [ ]:
train_hashes = pd.util.hash_pandas_object(X_train, index=False).to_numpy()
test_hashes = pd.util.hash_pandas_object(X_test, index=False).to_numpy()
shared_hashes = np.intersect1d(np.unique(train_hashes), np.unique(test_hashes))
overlap_rate = len(shared_hashes) / max(1, len(np.unique(test_hashes)))
overlap_df = pd.DataFrame([{
    "unique_train_vectors": len(np.unique(train_hashes)),
    "unique_test_vectors": len(np.unique(test_hashes)),
    "shared_unique_vectors": len(shared_hashes),
    "shared_fraction_unique_test": overlap_rate,
    "shared_percent_unique_test": 100 * overlap_rate,
    "split_strategy": SPLIT_STRATEGY,
}])
show_table(overlap_df, "Exact feature-vector overlap across train and test")
save_table(overlap_df, "train_test_exact_vector_overlap.csv")
if len(shared_hashes):
    warnings.warn("Exact feature vectors cross train/test; interpret random-row scores cautiously.")

def make_cv():
    if CV_STRATEGY == "stratified":
        return StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    if CV_STRATEGY == "duplicate_grouped":
        return StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    if CV_STRATEGY == "chronological":
        return TimeSeriesSplit(n_splits=CV_FOLDS)
    raise ValueError(CV_STRATEGY)

CV = make_cv()
if CV_STRATEGY == "chronological":
    X_train_search, y_train_search = X_train.copy(), y_train.copy()
    GROUPS_TRAIN_SEARCH = None
else:
    X_train_search, y_train_search = stratified_search_subset(
        X_train, y_train, SEARCH_MAX_ROWS
    )
    if CV_STRATEGY == "duplicate_grouped":
        group_series = pd.Series(GROUPS_TRAIN, index=X_train.index)
        GROUPS_TRAIN_SEARCH = group_series.loc[X_train_search.index].to_numpy()
    else:
        GROUPS_TRAIN_SEARCH = None

print(f"\nTraining rows: {len(X_train):,} | CV search subset: {len(X_train_search):,}")
print("Cross-validation strategy:", CV)

## 15. Reusable classes: train-fitted outlier clipping and fold-aware XGBoost weighting[¶](#15.-Reusable-classes:-train-fitted-outlier-clipping-and-fold-aware-XGBoost-weighting)

`QuantileClipper` learns bounds on the training fold only. `FoldAwareXGBClassifier` calculates `scale_pos_weight` inside each `.fit()`, so each CV fold uses its own training-label counts.

In [ ]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Fit per-feature quantile caps on the current training fold only."""
    def __init__(self, lower_quantile=0.01, upper_quantile=0.99):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile

    def fit(self, X, y=None):
        arr = np.asarray(X, dtype=float)
        self.lower_bounds_ = np.nanquantile(arr, self.lower_quantile, axis=0)
        self.upper_bounds_ = np.nanquantile(arr, self.upper_quantile, axis=0)
        self.n_features_in_ = arr.shape[1]
        return self

    def transform(self, X):
        arr = np.asarray(X, dtype=float)
        return np.clip(arr, self.lower_bounds_, self.upper_bounds_)

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray([f"x{i}" for i in range(self.n_features_in_)], dtype=object)
        return np.asarray(input_features, dtype=object)


class FoldAwareXGBClassifier(ClassifierMixin, BaseEstimator):
    """XGBoost wrapper that estimates scale_pos_weight from each fit fold."""
    def __init__(
        self, n_estimators=60, max_depth=6, learning_rate=0.1,
        min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, balance_classes=False, random_state=42, n_jobs=-1, device="cuda"
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.min_child_weight = min_child_weight
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.reg_lambda = reg_lambda
        self.balance_classes = balance_classes
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.device = device

    def fit(self, X, y):
        y = np.asarray(y)
        neg, pos = int(np.sum(y == 0)), int(np.sum(y == 1))
        pos_weight = neg / max(pos, 1) if self.balance_classes else 1.0
        self.estimator_ = XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", tree_method="hist", device=self.device,
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            learning_rate=self.learning_rate, min_child_weight=self.min_child_weight,
            subsample=self.subsample, colsample_bytree=self.colsample_bytree,
            reg_lambda=self.reg_lambda, scale_pos_weight=pos_weight,
            random_state=self.random_state, n_jobs=self.n_jobs,
        )
        X_fit = np.asarray(X, dtype=np.float32)
        self.estimator_.fit(X_fit, y)
        self.classes_ = self.estimator_.classes_
        self.n_features_in_ = getattr(self.estimator_, "n_features_in_", np.asarray(X).shape[1])
        return self

    def predict(self, X):
        return self.estimator_.predict(np.asarray(X, dtype=np.float32))

    def predict_proba(self, X):
        return self.estimator_.predict_proba(np.asarray(X, dtype=np.float32))

    @property
    def feature_importances_(self):
        return self.estimator_.feature_importances_


@dataclass(frozen=True)
class PreprocessingConfig:
    numeric_imputation: str = "median"
    add_missing_indicator: bool = False
    categorical_imputation: str = "most_frequent"
    protocol_encoding: str = "onehot"
    outlier_treatment: str = "none"
    numeric_transform: str = "none"
    scaling: str = "none"
    feature_selection: str = "none"
    selector_k: int = 20


@dataclass(frozen=True)
class ExperimentConfig:
    model_name: str
    preprocessing: PreprocessingConfig
    imbalance_strategy: str = "none"

    @property
    def key(self):
        return f"{self.model_name}|{self.imbalance_strategy}|{self.preprocessing}"

## 16. Factory functions: preprocessing, encoders, and model pipeline[¶](#16.-Factory-functions:-preprocessing,-encoders,-and-model-pipeline)

In [ ]:
def make_numeric_pipeline(
    imputation="median", add_indicator=False, outlier="none",
    numeric_transform="none", scaling="none"
):
    imputer = SimpleImputer(
        strategy=imputation,
        fill_value=0 if imputation == "constant" else None,
        add_indicator=add_indicator,
        keep_empty_features=True,
    )
    outlier_map = {
        "none": "passthrough",
        "clip_1_99": QuantileClipper(0.01, 0.99),
        "clip_05_995": QuantileClipper(0.005, 0.995),
    }
    transform_map = {
        "none": "passthrough",
        "yeo_johnson": PowerTransformer(method="yeo-johnson", standardize=False),
    }
    scaler_map = {
        "none": "passthrough",
        "standard": StandardScaler(),
        "robust": RobustScaler(),
        "minmax": MinMaxScaler(),
    }
    if outlier not in outlier_map or numeric_transform not in transform_map or scaling not in scaler_map:
        raise ValueError("Unknown imputation, outlier, distribution-transform, or scaling method.")
    return Pipeline([
        ("imputer", imputer),
        ("outlier", outlier_map[outlier]),
        ("transform", transform_map[numeric_transform]),
        ("scaler", scaler_map[scaling]),
    ])


def make_categorical_pipeline(imputation="most_frequent"):
    if imputation == "most_frequent":
        imputer = SimpleImputer(strategy="most_frequent", keep_empty_features=True)
    elif imputation == "constant":
        imputer = SimpleImputer(strategy="constant", fill_value="Missing", keep_empty_features=True)
    else:
        raise ValueError(imputation)
    return Pipeline([
        ("imputer", imputer),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)),
    ])


def make_preprocessor(config):
    transformers = []
    if NUMERIC_FEATURES:
        transformers.append((
            "numeric",
            make_numeric_pipeline(
                config.numeric_imputation, config.add_missing_indicator,
                config.outlier_treatment, config.numeric_transform, config.scaling
            ),
            NUMERIC_FEATURES,
        ))
    if CATEGORICAL_FEATURES:
        cat = (
            make_categorical_pipeline(config.categorical_imputation)
            if config.protocol_encoding == "onehot" else "drop"
        )
        transformers.append(("categorical", cat, CATEGORICAL_FEATURES))
    if not transformers:
        raise ValueError("No feature transformers remain.")
    return ColumnTransformer(
        transformers, remainder="drop", sparse_threshold=0.0,
        verbose_feature_names_out=False
    )


def make_classifier(model_name):
    if model_name == "Logistic Regression":
        return LogisticRegression(
            solver="lbfgs", max_iter=3000, class_weight=None,
            random_state=RANDOM_STATE
        )
    if model_name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=BASE_RF_ESTIMATORS, class_weight=None,
            n_jobs=MODEL_N_JOBS, random_state=RANDOM_STATE
        )
    if model_name == "XGBoost":
        if not XGBOOST_AVAILABLE:
            raise ImportError("Install through UV: uv add xgboost")
        return FoldAwareXGBClassifier(
            n_estimators=BASE_XGB_ESTIMATORS, max_depth=6, learning_rate=0.1,
            random_state=RANDOM_STATE, n_jobs=XGB_N_JOBS, device=XGB_DEVICE
        )
    raise ValueError(model_name)


def make_model_pipeline(model_name, preprocessing):
    return ImbPipeline([
        ("preprocessor", make_preprocessor(preprocessing)),
        ("sampler", "passthrough"),
        ("selector", "passthrough"),
        ("model", make_classifier(model_name)),
    ])


def evaluate_model(estimator, X_fit, y_fit, X_eval, y_eval, label):
    fitted = clone(estimator)
    fit_start = time.perf_counter()
    fitted.fit(X_fit, y_fit)
    fit_seconds = time.perf_counter() - fit_start
    pred_start = time.perf_counter()
    pred = fitted.predict(X_eval)
    prob = fitted.predict_proba(X_eval)[:, 1]
    predict_seconds = time.perf_counter() - pred_start
    metrics = calculate_metrics(y_eval, pred, prob)
    metrics.update({
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        "predict_ms_per_row": predict_seconds * 1000 / max(1, len(X_eval)),
    })
    return fitted, {"candidate": label, **metrics}

print("Factories, reusable configurations, and experiment runner are ready.")

## GPU execution policy

The XGBoost model is configured for CUDA and the notebook runs a tiny GPU smoke test before expensive experiments. The scikit-learn Logistic Regression and Random Forest estimators in this notebook are CPU implementations; setting an NVIDIA GPU does not move them to CUDA. GPU implementations of those algorithms require RAPIDS cuML, which NVIDIA supports on Linux/WSL2 rather than native Windows. This notebook keeps the original estimators comparable and accelerates XGBoost, the principal tree-boosting model.

In [ ]:
# GPU verification: this notebook does not silently label CPU training as GPU training.
# XGBoost >= 2.0 uses device="cuda" with tree_method="hist".
if not XGBOOST_AVAILABLE:
    raise RuntimeError("XGBoost is required for the requested CUDA run. Install it with: uv add \"xgboost>=2.0\"")

if XGBOOST_AVAILABLE:
    _xgb_major = int(str(xgb.__version__).split(".")[0])
    if _xgb_major < 2:
        raise RuntimeError(
            f"XGBoost {xgb.__version__} is too old for the device='cuda' API. "
            "Upgrade in this UV environment: uv add 'xgboost>=2.0'."
        )
    _build = xgb.build_info() if hasattr(xgb, "build_info") else {}
    _cuda_build = _build.get("USE_CUDA", None)
    if _cuda_build in (False, 0, "OFF", "off", "FALSE", "false"):
        raise RuntimeError(
            "This XGBoost package was built without CUDA support. Install a CUDA-enabled XGBoost build."
        )
    _probe = XGBClassifier(
        n_estimators=2, max_depth=2, tree_method="hist", device=XGB_DEVICE,
        eval_metric="logloss", n_jobs=1, verbosity=0,
    )
    _probe_X = np.asarray([[0, 0], [0, 1], [1, 0], [1, 1], [2, 1], [2, 2]], dtype=np.float32)
    _probe_y = np.asarray([0, 0, 0, 1, 1, 1], dtype=np.int8)
    _probe.fit(_probe_X, _probe_y)
    _probe_config = _probe.get_booster().save_config()
    if not re.search(r'"device"\s*:\s*"cuda', _probe_config):
        raise RuntimeError(
            "XGBoost accepted the CUDA parameter but its fitted configuration does not confirm CUDA. "
            "Stopping rather than silently running the main XGBoost search on CPU."
        )
    print(f"XGBoost GPU smoke test passed: version={xgb.__version__}, device={XGB_DEVICE}")
else:
    warnings.warn("XGBoost is unavailable; only CPU Logistic Regression and Random Forest will run.")

print("Estimator execution devices: Logistic Regression=CPU; Random Forest=CPU; XGBoost=CUDA (when available).")


## 17. Grid-search runner[¶](#17.-Grid-search-runner)

Every search uses the same scoring dictionary and CV protocol. Macro-F1 is the refit metric; Attack recall and FPR are inspected as additional operating characteristics.

In [ ]:
def run_grid_search(estimator, param_grid, X_fit, y_fit, label, groups=None, verbose=0):
    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=SCORING,
        refit=PRIMARY_SCORING,
        cv=CV,
        n_jobs=SEARCH_N_JOBS,
        verbose=verbose,
        error_score=np.nan,
        return_train_score=False,
        pre_dispatch="1*n_jobs",
    )
    kwargs = {"groups": groups} if groups is not None else {}
    start = time.perf_counter()
    search.fit(X_fit, y_fit, **kwargs)
    elapsed = time.perf_counter() - start
    results = pd.DataFrame(search.cv_results_)
    results["search_label"] = label
    results["search_seconds"] = elapsed
    results = results.sort_values("rank_test_f1_macro")
    return search, results, elapsed


def current_cv_groups():
    return GROUPS_TRAIN_SEARCH if CV_STRATEGY == "duplicate_grouped" else None


def mutual_info_score_func(X, y):
    return mutual_info_classif(X, y, random_state=RANDOM_STATE)

## 18. Stage 0 — default baseline benchmark[¶](#18.-Stage-0-%E2%80%94-default-baseline-benchmark)

Baseline: median imputation, one-hot Protocol, no outlier transformation, StandardScaler only for Logistic Regression, no feature selection, and no balancing.

In [ ]:
MODEL_NAMES = ["Logistic Regression", "Random Forest"]
if XGBOOST_AVAILABLE:
    MODEL_NAMES.append("XGBoost")

BASE_PREPROCESSING = {
    name: PreprocessingConfig(
        numeric_imputation="median",
        add_missing_indicator=False,
        categorical_imputation="most_frequent",
        protocol_encoding="onehot",
        outlier_treatment="none",
        numeric_transform="none",
        scaling="standard" if name == "Logistic Regression" else "none",
    )
    for name in MODEL_NAMES
}

baseline_rows = []
baseline_fitted = {}
for name in MODEL_NAMES:
    print(f"\nBaseline benchmark: {name}")
    pipe = make_model_pipeline(name, BASE_PREPROCESSING[name])
    cv_scores = cross_validate(
        pipe, X_train_search, y_train_search,
        scoring=SCORING, cv=CV, n_jobs=SEARCH_N_JOBS,
        groups=current_cv_groups(), return_train_score=False
    )
    fitted, val = evaluate_model(pipe, X_train, y_train, X_val, y_val, f"{name}|baseline")
    baseline_fitted[name] = fitted
    baseline_rows.append({
        "model": name,
        "cv_f1_macro_mean": float(np.nanmean(cv_scores["test_f1_macro"])),
        "cv_f1_macro_std": float(np.nanstd(cv_scores["test_f1_macro"], ddof=1)),
        "cv_roc_auc_mean": float(np.nanmean(cv_scores["test_roc_auc"])) if "test_roc_auc" in cv_scores else np.nan,
        "cv_average_precision_mean": float(np.nanmean(cv_scores["test_average_precision"])) if "test_average_precision" in cv_scores else np.nan,
        **{k: v for k, v in val.items() if k != "candidate"},
    })

baseline_results_df = rank_validation_results(pd.DataFrame(baseline_rows))
show_table(baseline_results_df, "Stage 0 — baseline validation results", columns=["model", "cv_f1_macro_mean", "cv_f1_macro_std", "cv_roc_auc_mean", "cv_average_precision_mean", "accuracy", "precision_attack", "recall_attack", "f1_macro", "false_positive_rate", "false_negative_rate", "fit_seconds"])
save_table(baseline_results_df, "stage0_baseline_results.csv")

## 19. Stage 1 — imbalance strategy search[¶](#19.-Stage-1-%E2%80%94-imbalance-strategy-search)

Compare none, class weighting, RandomOverSampler, and RandomUnderSampler. Resampling is inside the imbalanced-learn pipeline and therefore runs only on each CV training fold. We do not combine weights and sampling in one candidate.

In [ ]:
def make_imbalance_grid(model_name):
    if model_name == "Logistic Regression":
        weight_grid = {"sampler": ["passthrough"], "model__class_weight": [None, "balanced"]}
        sampler_grid = {
            "sampler": [RandomOverSampler(random_state=RANDOM_STATE),
                        RandomUnderSampler(random_state=RANDOM_STATE)],
            "model__class_weight": [None],
        }
    elif model_name == "Random Forest":
        weight_grid = {"sampler": ["passthrough"], "model__class_weight": [None, "balanced_subsample"]}
        sampler_grid = {
            "sampler": [RandomOverSampler(random_state=RANDOM_STATE),
                        RandomUnderSampler(random_state=RANDOM_STATE)],
            "model__class_weight": [None],
        }
    else:
        weight_grid = {"sampler": ["passthrough"], "model__balance_classes": [False, True]}
        sampler_grid = {
            "sampler": [RandomOverSampler(random_state=RANDOM_STATE),
                        RandomUnderSampler(random_state=RANDOM_STATE)],
            "model__balance_classes": [False],
        }
    return [weight_grid, sampler_grid]

imbalance_searches = {}
imbalance_cv_frames = []
imbalance_val_rows = []
imbalance_fitted_pipelines = {}

for name in MODEL_NAMES:
    base = make_model_pipeline(name, BASE_PREPROCESSING[name])
    search, cv_frame, elapsed = run_grid_search(
        base, make_imbalance_grid(name), X_train_search, y_train_search,
        f"imbalance|{name}", groups=current_cv_groups()
    )
    imbalance_searches[name] = search
    cv_frame["model"] = name
    imbalance_cv_frames.append(cv_frame)

    fitted, val = evaluate_model(
        search.best_estimator_, X_train, y_train, X_val, y_val, f"{name}|best_imbalance"
    )
    imbalance_fitted_pipelines[name] = fitted
    imbalance_val_rows.append({
        **{k: v for k, v in val.items() if k != "candidate"},
        "model": name,
        "best_params": repr(search.best_params_),
        "cv_best_f1_macro": float(search.best_score_),
        "search_seconds": elapsed,
    })

imbalance_cv_df = pd.concat(imbalance_cv_frames, ignore_index=True)
imbalance_validation_df = rank_validation_results(pd.DataFrame(imbalance_val_rows))
show_table(imbalance_validation_df, "Stage 1 — best imbalance strategy per model", columns=["model", "accuracy", "precision_attack", "recall_attack", "f1_macro", "average_precision", "false_positive_rate", "false_negative_rate", "cv_best_f1_macro", "search_seconds", "best_params"])
save_table(imbalance_cv_df, "stage1_imbalance_cv_candidates.csv")
save_table(imbalance_validation_df, "stage1_imbalance_validation.csv")

## 20. Stage 2 — preprocessing sensitivity grids[¶](#20.-Stage-2-%E2%80%94-preprocessing-sensitivity-grids)

Each factor is searched separately for each classifier, holding the other selected components fixed.

- Numeric imputation: median / mean / constant zero, with or without a missing indicator.
- Outliers: none / quantile clipping; Yeo-Johnson is also tested for Logistic Regression.
- Protocol: one-hot / feature ablation (drop).
- Scaling: none / Standard / Robust / MinMax for Logistic Regression only.
- Feature selection: none / ANOVA top-20 / ANOVA top-40; mutual information is opt-in.
- Categorical imputation is searched only if Protocol actually has missing values.

In [ ]:
def selector_candidates(max_features):
    """Candidate selectors fit inside each CV fold; no full-dataset feature selection."""
    opts = ["passthrough"]
    opts.append(VarianceThreshold(threshold=0.0))
    for k in [10, 20, 40]:
        if 1 <= k < max_features:
            opts.append(SelectKBest(score_func=f_classif, k=k))
    if INCLUDE_MUTUAL_INFO_IN_SEARCH:
        for k in [10, 20]:
            if 1 <= k < max_features:
                opts.append(SelectKBest(score_func=mutual_info_score_func, k=k))
    return opts


def make_preprocessing_factor_grids(model_name):
    encoded_upper = len(NUMERIC_FEATURES) + (
        int(X_train[PROTOCOL_COLUMN].nunique(dropna=True))
        if PROTOCOL_COLUMN in CATEGORICAL_FEATURES else 0
    )
    grids = {
        "numeric_imputation": {
            "preprocessor__numeric__imputer__strategy": ["median", "mean", "constant"],
            "preprocessor__numeric__imputer__add_indicator": [False, True],
        },
        "outlier_treatment": {
            "preprocessor__numeric__outlier": [
                "passthrough", QuantileClipper(0.01, 0.99), QuantileClipper(0.005, 0.995)
            ],
        },
        "feature_selection": {"selector": selector_candidates(encoded_upper)},
    }
    if model_name == "Logistic Regression":
        grids["numeric_transform"] = {
            "preprocessor__numeric__transform": [
                "passthrough", PowerTransformer(method="yeo-johnson", standardize=False)
            ]
        }
        grids["scaling"] = {
            "preprocessor__numeric__scaler": [
                StandardScaler(), RobustScaler(), MinMaxScaler(), "passthrough"
            ]
        }
    # Protocol is low-cardinality nominal: one-hot is the meaningful encoding.
    # Dropping it is retained as a feature-ablation experiment, not an alternative encoder.
    if CATEGORICAL_FEATURES:
        grids["protocol_handling"] = {
            "preprocessor__categorical": [make_categorical_pipeline("most_frequent"), "drop"]
        }
        if X_train[CATEGORICAL_FEATURES].isna().any().any():
            grids["categorical_imputation"] = {
                "preprocessor__categorical__imputer__strategy": ["most_frequent", "constant"],
            }
    return grids


preprocessing_searches = {}
preprocessing_cv_frames = []
preprocessing_winners = {}
preprocessing_validation_rows = []

for name in MODEL_NAMES:
    print(f"\nPreprocessing sensitivity: {name}")
    base = clone(imbalance_searches[name].best_estimator_)
    winners = {}

    for factor, grid in make_preprocessing_factor_grids(name).items():
        print("  Searching:", factor)
        # If Protocol is dropped, its conditional imputation parameter cannot be merged later.
        search, cv_frame, elapsed = run_grid_search(
            clone(base), grid, X_train_search, y_train_search,
            f"preprocessing|{name}|{factor}", groups=current_cv_groups()
        )
        preprocessing_searches[(name, factor)] = search
        winners[factor] = search.best_params_
        cv_frame["model"] = name
        cv_frame["factor"] = factor
        preprocessing_cv_frames.append(cv_frame)

        fitted, val = evaluate_model(
            search.best_estimator_, X_train, y_train, X_val, y_val,
            f"{name}|{factor}"
        )
        preprocessing_validation_rows.append({
            **{k: v for k, v in val.items() if k != "candidate"},
            "model": name,
            "factor": factor,
            "cv_best_f1_macro": float(search.best_score_),
            "best_params": repr(search.best_params_),
            "search_seconds": elapsed,
        })

    preprocessing_winners[name] = winners

preprocessing_cv_df = pd.concat(preprocessing_cv_frames, ignore_index=True)
preprocessing_validation_df = pd.DataFrame(preprocessing_validation_rows)
show_table(preprocessing_validation_df.sort_values(["model", "f1_macro"], ascending=[True, False]), "Stage 2 — preprocessing sensitivity (validation)", columns=["model", "factor", "cv_best_f1_macro", "accuracy", "precision_attack", "recall_attack", "f1_macro", "false_positive_rate", "false_negative_rate", "search_seconds", "best_params"], max_rows=40)
save_table(preprocessing_cv_df, "stage2_preprocessing_cv_candidates.csv")
save_table(preprocessing_validation_df, "stage2_preprocessing_validation.csv")

## 21. Stage 2b — merge and validate the combined preprocessing candidate[¶](#21.-Stage-2b-%E2%80%94-merge-and-validate-the-combined-preprocessing-candidate)

Individually best choices may interact. We merge their best CV parameters, refit the combined candidate on `X_train`, and compare it with the imbalance-selected baseline on `X_val`. The combined configuration is kept only if its validation ranking improves.

In [ ]:
selected_pipeline_by_model = {}
combined_validation_rows = []
combined_params_by_model = {}

for name in MODEL_NAMES:
    baseline_pipe = clone(imbalance_searches[name].best_estimator_)
    combined_pipe = clone(baseline_pipe)
    merged = {}
    for factor_params in preprocessing_winners[name].values():
        merged.update(factor_params)

    # A categorical-imputation choice only applies when Protocol remains in the model.
    if merged.get("preprocessor__categorical") == "drop":
        merged = {k: v for k, v in merged.items()
                  if not k.startswith("preprocessor__categorical__")}

    if merged:
        combined_pipe.set_params(**merged)

    baseline_fit, baseline_val = evaluate_model(
        baseline_pipe, X_train, y_train, X_val, y_val, f"{name}|imbalance_baseline"
    )
    combined_fit, combined_val = evaluate_model(
        combined_pipe, X_train, y_train, X_val, y_val, f"{name}|combined_pipeline"
    )

    comparison = pd.DataFrame([
        {"candidate": "imbalance baseline", **{k:v for k,v in baseline_val.items() if k!="candidate"}},
        {"candidate": "combined preprocessing", **{k:v for k,v in combined_val.items() if k!="candidate"}},
    ])
    show_table(comparison[["candidate", "f1_macro", "recall_attack", "false_positive_rate", "average_precision"]], f"{name} — baseline vs combined pipeline")

    winner = rank_validation_results(comparison).iloc[0]["candidate"]
    selected_pipeline_by_model[name] = combined_fit if winner == "combined preprocessing" else baseline_fit
    combined_params_by_model[name] = merged

    combined_validation_rows.extend([
        {"model": name, "candidate_type": "imbalance baseline", **{k:v for k,v in baseline_val.items() if k!="candidate"}},
        {"model": name, "candidate_type": "combined preprocessing", **{k:v for k,v in combined_val.items() if k!="candidate"}},
    ])

combined_validation_df = pd.DataFrame(combined_validation_rows)
show_table(combined_validation_df.sort_values(["model", "f1_macro"], ascending=[True, False]), "Stage 2B — combined pipeline comparison", columns=["model", "candidate_type", "accuracy", "precision_attack", "recall_attack", "f1_macro", "false_positive_rate", "average_precision", "fit_seconds"])
save_table(combined_validation_df, "stage2b_combined_validation.csv")

## 22. Stage 3 — model hyperparameter GridSearchCV[¶](#22.-Stage-3-%E2%80%94-model-hyperparameter-GridSearchCV)

The pipeline configuration is now fixed for each model. Tune only model-level parameters.

- Logistic Regression: `C`.
- Random Forest: `n_estimators`, `max_depth`, `min_samples_leaf`.
- XGBoost: `n_estimators`, `max_depth`, `learning_rate`, `min_child_weight`.

These are compact, discrete grids. If the ranges are broadened substantially, use RandomizedSearchCV or Bayesian optimization rather than making the exhaustive grid explode.

In [ ]:
def hyperparameter_grid(model_name):
    """Compact discrete model-level grid. Expand only after profiling compute cost."""
    if model_name == "Logistic Regression":
        return {"model__C": [0.1, 1.0, 10.0]}
    if model_name == "Random Forest":
        return {
            "model__n_estimators": RF_SEARCH_N_ESTIMATORS,
            "model__max_depth": [None, 20],
            "model__min_samples_leaf": [1, 3],
        }
    if model_name == "XGBoost":
        return {
            "model__n_estimators": XGB_SEARCH_N_ESTIMATORS,
            "model__max_depth": [3, 5],
            "model__learning_rate": [0.05, 0.1],
            "model__min_child_weight": [1, 5],
        }
    raise ValueError(model_name)

hyperparameter_searches = {}
hyperparameter_cv_frames = []
tuned_validation_rows = []
tuned_pipeline_by_model = {}

for name in MODEL_NAMES:
    print(f"\nHyperparameter tuning: {name}")
    search, cv_frame, elapsed = run_grid_search(
        clone(selected_pipeline_by_model[name]),
        hyperparameter_grid(name),
        X_train_search, y_train_search,
        f"hyperparameters|{name}", groups=current_cv_groups(), verbose=0
    )
    hyperparameter_searches[name] = search
    print(
        f"  Evaluated {len(cv_frame):,} candidate configurations × {CV_FOLDS} folds "
        f"= {len(cv_frame) * CV_FOLDS:,} fits in {elapsed:.1f} seconds."
    )
    cv_frame["model"] = name
    hyperparameter_cv_frames.append(cv_frame)

    fitted, val = evaluate_model(
        search.best_estimator_, X_train, y_train, X_val, y_val, f"{name}|tuned"
    )
    tuned_pipeline_by_model[name] = fitted
    tuned_validation_rows.append({
        "model": name,
        "best_params": repr(search.best_params_),
        "cv_best_f1_macro": float(search.best_score_),
        "search_seconds": elapsed,
        **{k: v for k, v in val.items() if k != "candidate"},
    })

hyperparameter_cv_df = pd.concat(hyperparameter_cv_frames, ignore_index=True)
tuned_validation_df = rank_validation_results(pd.DataFrame(tuned_validation_rows))
show_table(tuned_validation_df, "Stage 3 — tuned models on validation data", columns=["model", "cv_best_f1_macro", "accuracy", "precision_attack", "recall_attack", "f1_macro", "false_positive_rate", "false_negative_rate", "roc_auc", "average_precision", "mcc", "search_seconds", "best_params"])
save_table(hyperparameter_cv_df, "stage3_hyperparameter_cv_candidates.csv")
save_table(tuned_validation_df, "stage3_tuned_validation.csv")

## 23. Select the validation champion and a tree model for SHAP[¶](#23.-Select-the-validation-champion-and-a-tree-model-for-SHAP)

In [ ]:
BEST_MODEL_NAME = str(tuned_validation_df.iloc[0]["model"])
BEST_VALIDATION_PIPELINE = tuned_pipeline_by_model[BEST_MODEL_NAME]
print("Overall validation champion:", BEST_MODEL_NAME)

tree_rows = tuned_validation_df[tuned_validation_df["model"].isin(["Random Forest", "XGBoost"])]
if tree_rows.empty:
    BEST_TREE_NAME = None
    BEST_TREE_PIPELINE = None
else:
    BEST_TREE_NAME = str(rank_validation_results(tree_rows).iloc[0]["model"])
    BEST_TREE_PIPELINE = tuned_pipeline_by_model[BEST_TREE_NAME]
    print("Tree model for SHAP:", BEST_TREE_NAME)

## 24. SHAP global and local explanations[¶](#24.-SHAP-global-and-local-explanations)

SHAP is calculated on a reproducible sample from the training partition. For Random Forest, SHAP may return a class dimension; for XGBoost the default raw output is usually a margin/log-odds value. We normalize the output shape for feature ranking.

SHAP contributions explain model output; they do not establish causal relationships.

In [ ]:
shap_importance = pd.Series(dtype=float)
if BEST_TREE_PIPELINE is not None and SHAP_AVAILABLE:
    pre = BEST_TREE_PIPELINE.named_steps["preprocessor"]
    selector = BEST_TREE_PIPELINE.named_steps["selector"]
    model_step = BEST_TREE_PIPELINE.named_steps["model"]
    tree_model = model_step.estimator_ if isinstance(model_step, FoldAwareXGBClassifier) else model_step

    rng = np.random.default_rng(RANDOM_STATE)
    n_shap = min(SHAP_SAMPLE_SIZE, len(X_train))
    rows = rng.choice(len(X_train), size=n_shap, replace=False)
    X_shap_raw = X_train.iloc[rows]

    X_shap_pre = pre.transform(X_shap_raw)
    transformed_names = np.asarray(pre.get_feature_names_out(), dtype=object)
    if selector != "passthrough":
        X_shap_sel = selector.transform(X_shap_pre)
        transformed_names = transformed_names[selector.get_support()]
    else:
        X_shap_sel = X_shap_pre

    explainer = shap.TreeExplainer(tree_model, feature_perturbation="tree_path_dependent")
    raw_values = explainer.shap_values(X_shap_sel)
    if isinstance(raw_values, list):
        shap_values = np.asarray(raw_values[1] if len(raw_values) > 1 else raw_values[0])
    else:
        shap_values = np.asarray(raw_values)
        if shap_values.ndim == 3:
            # SHAP versions may return (rows, features, classes) or (classes, rows, features).
            if shap_values.shape[0] == len(X_shap_sel) and shap_values.shape[1] == len(transformed_names):
                shap_values = shap_values[:, :, 1]
            elif shap_values.shape[-1] == len(transformed_names):
                shap_values = shap_values[1, :, :]

    if shap_values.ndim != 2 or shap_values.shape[1] != len(transformed_names):
        raise ValueError(f"SHAP shape {shap_values.shape} does not match {len(transformed_names)} features.")

    shap_importance = pd.Series(
        np.abs(shap_values).mean(axis=0), index=transformed_names, name="mean_abs_shap"
    ).sort_values(ascending=False)
    show_table(shap_importance.head(30).to_frame().reset_index(names="feature"), "SHAP global feature importance — top 30", max_rows=30)
    shap_importance.to_csv(TABLES_DIR / "shap_global_feature_importance.csv")

    shap.summary_plot(
        shap_values, X_shap_sel, feature_names=transformed_names.tolist(),
        max_display=20, show=False
    )
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "shap_summary_beeswarm.png", dpi=180, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 7))
    shap_importance.head(20).sort_values().plot(kind="barh", ax=ax)
    ax.set_title(f"Top SHAP features — {BEST_TREE_NAME}")
    ax.set_xlabel("Mean absolute SHAP value")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "shap_global_top20.png", dpi=180)
    plt.show()

    expected = np.asarray(explainer.expected_value).reshape(-1)
    base_value = float(expected[1] if len(expected) > 1 else expected[0])
    local_exp = shap.Explanation(
        values=shap_values[0], base_values=base_value,
        data=np.asarray(X_shap_sel)[0], feature_names=transformed_names.tolist()
    )
    shap.plots.waterfall(local_exp, max_display=15)
else:
    print("SHAP skipped: tree model or SHAP library is unavailable.")

## 25. SHAP-guided feature reduction[¶](#25.-SHAP-guided-feature-reduction)

SHAP ranks features from training rows only. We compare pre-specified top-k subsets (10/20/30) on validation. The feature count is selected using validation Macro-F1, with a 0.01 tolerance from the full-feature tree model. The test set remains untouched.

In [ ]:
shap_reduction_validation_df = pd.DataFrame()
reduced_validation_models = {}
selected_reduced_features = []
SELECTED_SHAP_K = None

if BEST_TREE_PIPELINE is not None and SHAP_AVAILABLE and not shap_importance.empty:
    names = shap_importance.index.tolist()
    name_to_idx = {name: i for i, name in enumerate(names)}
    top_ks = sorted({min(int(k), len(names)) for k in TOP_K_FEATURES if int(k) > 0})
    top_ks = [k for k in top_ks if k < len(names)]
    if not top_ks and len(names) > 0:
        top_ks = [len(names)]

    pre = BEST_TREE_PIPELINE.named_steps["preprocessor"]
    selector = BEST_TREE_PIPELINE.named_steps["selector"]
    sampler_template = BEST_TREE_PIPELINE.named_steps["sampler"]
    model_template = BEST_TREE_PIPELINE.named_steps["model"]

    X_train_pre = pre.transform(X_train)
    X_val_pre = pre.transform(X_val)
    X_train_sel = selector.transform(X_train_pre) if selector != "passthrough" else X_train_pre
    X_val_sel = selector.transform(X_val_pre) if selector != "passthrough" else X_val_pre

    reduction_rows = []
    for k in top_ks:
        selected_names = names[:k]
        selected_idx = np.asarray([name_to_idx[n] for n in selected_names], dtype=int)
        X_tr_k = np.asarray(X_train_sel)[:, selected_idx]
        X_va_k = np.asarray(X_val_sel)[:, selected_idx]

        if sampler_template == "passthrough":
            X_fit, y_fit = X_tr_k, y_train
        else:
            sampler = clone(sampler_template)
            X_fit, y_fit = sampler.fit_resample(X_tr_k, y_train)

        model_k = clone(model_template).fit(X_fit, y_fit)
        pred = model_k.predict(X_va_k)
        prob = model_k.predict_proba(X_va_k)[:, 1]
        reduction_rows.append({
            "features_used": k,
            "feature_fraction": k / len(names),
            **calculate_metrics(y_val, pred, prob),
        })
        reduced_validation_models[k] = {
            "model": model_k, "indices": selected_idx, "features": selected_names
        }

    shap_reduction_validation_df = pd.DataFrame(reduction_rows).sort_values("features_used")
    show_table(shap_reduction_validation_df, "SHAP top-k feature reduction — validation", columns=["features_used", "feature_fraction", "accuracy", "precision_attack", "recall_attack", "f1_attack", "f1_macro", "roc_auc", "average_precision", "false_positive_rate", "false_negative_rate"])
    save_table(shap_reduction_validation_df, "shap_feature_reduction_validation.csv")

    full_pred = BEST_TREE_PIPELINE.predict(X_val)
    full_prob = BEST_TREE_PIPELINE.predict_proba(X_val)[:, 1]
    full_val_metrics = calculate_metrics(y_val, full_pred, full_prob)
    eligible = shap_reduction_validation_df[
        shap_reduction_validation_df["f1_macro"] >= full_val_metrics["f1_macro"] - 0.01
    ]
    if len(eligible):
        SELECTED_SHAP_K = int(eligible.sort_values("features_used").iloc[0]["features_used"])
    else:
        SELECTED_SHAP_K = int(rank_validation_results(shap_reduction_validation_df).iloc[0]["features_used"])

    selected_reduced_features = reduced_validation_models[SELECTED_SHAP_K]["features"]
    print("Selected SHAP top-k:", SELECTED_SHAP_K)
    print(selected_reduced_features)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(shap_reduction_validation_df["features_used"], shap_reduction_validation_df["f1_macro"],
            marker="o", label="SHAP reduced")
    ax.axhline(full_val_metrics["f1_macro"], linestyle="--", label="Full feature model")
    ax.set_xlabel("Number of features")
    ax.set_ylabel("Validation Macro-F1")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "shap_feature_reduction_validation.png", dpi=180)
    plt.show()
else:
    print("SHAP reduction skipped.")

## 26. Final test evaluation[¶](#26.-Final-test-evaluation)

Refit each tuned model on Train + Validation, then evaluate once on the untouched Test partition. Test metrics are for reporting only; do not return to the test set to pick methods or hyperparameters.

In [ ]:
X_train_validation = pd.concat([X_train, X_val], axis=0)
y_train_validation = pd.concat([y_train, y_val], axis=0)

final_fitted_models = {}
test_prediction_store = {}
test_rows = []

for name in MODEL_NAMES:
    final_model = clone(hyperparameter_searches[name].best_estimator_)
    fit_start = time.perf_counter()
    final_model.fit(X_train_validation, y_train_validation)
    final_fit_seconds = time.perf_counter() - fit_start
    pred_start = time.perf_counter()
    pred = final_model.predict(X_test)
    prob = final_model.predict_proba(X_test)[:, 1]
    predict_seconds = time.perf_counter() - pred_start
    metrics = calculate_metrics(y_test, pred, prob)
    metrics.update({
        "final_fit_seconds": final_fit_seconds,
        "test_predict_seconds": predict_seconds,
        "predict_ms_per_row": predict_seconds * 1000 / max(1, len(X_test)),
    })
    test_rows.append({
        "model": name,
        "best_hyperparameters": repr(hyperparameter_searches[name].best_params_),
        **metrics,
    })
    final_fitted_models[name] = final_model
    test_prediction_store[name] = (pred, prob)

final_test_results_df = pd.DataFrame(test_rows).sort_values("f1_macro", ascending=False)
show_table(final_test_results_df, "Final in-domain test results — full-feature models", columns=["model", "accuracy", "precision_attack", "recall_attack", "f1_attack", "f1_macro", "f1_weighted", "balanced_accuracy", "roc_auc", "average_precision", "mcc", "false_positive_rate", "false_negative_rate", "TN", "FP", "FN", "TP", "final_fit_seconds", "test_predict_seconds"])
save_table(final_test_results_df, "final_test_model_comparison.csv")

## 27. Refit and test the selected SHAP-reduced tree model[¶](#27.-Refit-and-test-the-selected-SHAP-reduced-tree-model)

The preprocessor and optional filter selector are refitted on Train + Validation only. The selected SHAP features are then used to fit the reduced model. The test partition is only transformed and predicted.

In [ ]:
shap_reduced_test_record = None
shap_reduced_test_pred = None
shap_reduced_test_prob = None

if BEST_TREE_NAME is not None and SELECTED_SHAP_K is not None and selected_reduced_features:
    selected_pipeline = hyperparameter_searches[BEST_TREE_NAME].best_estimator_

    final_pre = clone(selected_pipeline.named_steps["preprocessor"]).fit(
        X_train_validation, y_train_validation
    )
    X_tv_pre = final_pre.transform(X_train_validation)
    X_test_pre = final_pre.transform(X_test)
    final_names = np.asarray(final_pre.get_feature_names_out(), dtype=object)

    selector_template = selected_pipeline.named_steps["selector"]
    if selector_template != "passthrough":
        final_selector = clone(selector_template).fit(X_tv_pre, y_train_validation)
        X_tv_sel = final_selector.transform(X_tv_pre)
        X_test_sel = final_selector.transform(X_test_pre)
        final_names = final_names[final_selector.get_support()]
    else:
        final_selector = "passthrough"
        X_tv_sel, X_test_sel = X_tv_pre, X_test_pre

    name_to_index = {name: i for i, name in enumerate(final_names)}
    missing_names = [n for n in selected_reduced_features if n not in name_to_index]
    if missing_names:
        raise ValueError(f"Feature names changed during refit: {missing_names}")
    selected_idx = np.asarray([name_to_index[n] for n in selected_reduced_features], dtype=int)

    X_tv_k = np.asarray(X_tv_sel)[:, selected_idx]
    X_test_k = np.asarray(X_test_sel)[:, selected_idx]

    sampler_template = selected_pipeline.named_steps["sampler"]
    if sampler_template == "passthrough":
        X_fit, y_fit = X_tv_k, y_train_validation
    else:
        final_sampler = clone(sampler_template)
        X_fit, y_fit = final_sampler.fit_resample(X_tv_k, y_train_validation)

    reduced_final_model = clone(selected_pipeline.named_steps["model"]).fit(X_fit, y_fit)
    shap_reduced_test_pred = reduced_final_model.predict(X_test_k)
    shap_reduced_test_prob = reduced_final_model.predict_proba(X_test_k)[:, 1]

    shap_reduced_test_record = {
        "model": f"{BEST_TREE_NAME} (SHAP top-{SELECTED_SHAP_K})",
        "feature_count": len(selected_reduced_features),
        **calculate_metrics(y_test, shap_reduced_test_pred, shap_reduced_test_prob),
    }
    show_table(pd.DataFrame([shap_reduced_test_record]), "Final in-domain test result — SHAP-reduced model")
    save_table(pd.DataFrame([shap_reduced_test_record]), "shap_reduced_model_test_metrics.csv")
    test_prediction_store[shap_reduced_test_record["model"]] = (
        shap_reduced_test_pred, shap_reduced_test_prob
    )
else:
    print("No SHAP-reduced final model was selected.")

## Complete model diagnostics

For each model, the training and validation metrics below come from the tuned model fitted on the training partition only. The test metrics come from the final model refitted on train + validation, evaluated once on the untouched test partition. This avoids presenting a test-refitted model's resubstitution scores as independent train/validation performance.

In [ ]:
# Full train / validation / test diagnostics for every model.
# Train/validation scores use the tuned pipeline fitted on TRAIN only.
# Test scores use the final pipeline refitted on TRAIN + VALIDATION; test remains untouched.
SPLIT_METRIC_ROWS = []
SPLIT_CLASS_REPORT_ROWS = []
SPLIT_CONFUSION_MATRICES = {}

for model_name in MODEL_NAMES:
    validation_fit_model = tuned_pipeline_by_model[model_name]  # fitted on X_train only
    final_model = final_fitted_models[model_name]              # refitted on train + validation
    evaluations = [
        ("train", validation_fit_model, X_train, y_train),
        ("validation", validation_fit_model, X_val, y_val),
        ("test", final_model, X_test, y_test),
    ]
    for split_name, fitted_model, X_part, y_part in evaluations:
        pred_part = fitted_model.predict(X_part)
        prob_part = fitted_model.predict_proba(X_part)[:, 1]
        metrics_part = calculate_metrics(y_part, pred_part, prob_part)
        SPLIT_METRIC_ROWS.append({
            "model": model_name, "split": split_name,
            "model_fit_scope": "train_only" if split_name in ("train", "validation") else "train_plus_validation",
            "rows": int(len(y_part)), **metrics_part,
        })
        report = classification_report(
            y_part, pred_part, labels=[0, 1], target_names=["Benign", "Attack"],
            output_dict=True, zero_division=0,
        )
        for class_name in ["Benign", "Attack"]:
            SPLIT_CLASS_REPORT_ROWS.append({
                "model": model_name, "split": split_name, "class": class_name,
                **report[class_name],
            })
        SPLIT_CONFUSION_MATRICES[(model_name, split_name)] = confusion_matrix(
            y_part, pred_part, labels=[0, 1]
        )

model_split_metrics_df = pd.DataFrame(SPLIT_METRIC_ROWS)
model_split_class_report_df = pd.DataFrame(SPLIT_CLASS_REPORT_ROWS)
show_table(
    model_split_metrics_df,
    "Complete model metrics — train / validation / test",
    columns=["model", "split", "model_fit_scope", "rows", "accuracy", "precision_attack", "recall_attack", "specificity_benign", "f1_attack", "f2_attack", "f1_macro", "f1_weighted", "balanced_accuracy", "roc_auc", "average_precision", "mcc", "cohen_kappa", "g_mean", "false_positive_rate", "false_negative_rate", "TN", "FP", "FN", "TP"],
    max_rows=100,
)
show_table(model_split_class_report_df, "Per-class precision / recall / F-score — all models and splits", max_rows=100)
save_table(model_split_metrics_df, "v3_model_metrics_train_validation_test.csv")
save_table(model_split_class_report_df, "v3_model_classification_reports_train_validation_test.csv")

n_models = len(MODEL_NAMES)
fig, axes = plt.subplots(n_models, 3, figsize=(15, max(4, 4 * n_models)), squeeze=False)
for row_idx, model_name in enumerate(MODEL_NAMES):
    for col_idx, split_name in enumerate(["train", "validation", "test"]):
        ConfusionMatrixDisplay(
            confusion_matrix=SPLIT_CONFUSION_MATRICES[(model_name, split_name)],
            display_labels=["Benign", "Attack"],
        ).plot(ax=axes[row_idx, col_idx], values_format="d", colorbar=False)
        axes[row_idx, col_idx].set_title(f"{model_name} — {split_name}")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "v3_all_models_train_validation_test_confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.show()

print("Interpretation note: train/validation rows use a model fitted on train only; test rows use the final refit on train+validation.")


## 28. Per-class reports, confusion matrices, ROC and Precision–Recall curves[¶](#28.-Per-class-reports,-confusion-matrices,-ROC-and-Precision%E2%80%93Recall-curves)

The selected SHAP-reduced model is included in the plots when it was successfully constructed.

In [ ]:
per_class_rows = []
for name, (pred, prob) in test_prediction_store.items():
    report = classification_report(
        y_test, pred, labels=[0, 1], target_names=["Benign", "Attack"],
        output_dict=True, zero_division=0
    )
    for class_name in ["Benign", "Attack"]:
        row = report[class_name].copy()
        row.update({"model": name, "class": class_name})
        per_class_rows.append(row)

per_class_df = pd.DataFrame(per_class_rows)
show_table(per_class_df, "Per-class test metrics", columns=["model", "class", "precision", "recall", "f1-score", "support"])
save_table(per_class_df, "test_per_class_metrics.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, (pred, prob) in test_prediction_store.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, prob, name=name, ax=axes[1])
axes[0].set_title("Test ROC curves")
axes[1].set_title("Test Precision–Recall curves")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_roc_pr_curves.png", dpi=180)
plt.show()

n_models = len(test_prediction_store)
n_cols = 2
n_rows = int(np.ceil(n_models / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows))
axes = np.asarray(axes).reshape(-1)
for ax, (name, (pred, _)) in zip(axes, test_prediction_store.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, labels=[0, 1], display_labels=["Benign", "Attack"],
        values_format="d", cmap="Blues", ax=ax, colorbar=False
    )
    ax.set_title(name)
for ax in axes[n_models:]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_confusion_matrices.png", dpi=180)
plt.show()

## 28A. Save the selected model and reusable prediction artifacts[¶](#28A.-Save-the-selected-model-and-reusable-prediction-artifacts)

The full-feature model selected using **validation data** is refitted on the source training + validation partitions in Section 27. This section saves that fitted end-to-end pipeline, including its fitted preprocessing, selector, sampler configuration, and classifier. The final CIC-IDS2017 test partition is not used to choose the model.

Two artifacts are written under `models/artifacts/`:

1. `best_cicids2017_full_pipeline.joblib` — the validation-selected full-feature model, ready for inference on data with the same input feature schema.
2. `best_cicids2017_shap_reduced_bundle.joblib` — the SHAP-reduced tree model together with its fitted preprocessor, selector, selected feature indices, and required raw input columns, when that model exists.

The accompanying JSON metadata records the feature schema and target mapping. These files are generated when this cell is executed on your computer; they are not pre-trained model files included with this notebook. Because this notebook defines custom estimator/transformer classes, load the artifact in a Python session where those class definitions have been executed, or move the classes into an importable module before using the artifact from a standalone script.

In [ ]:
MODEL_ARTIFACT_DIR = PROJECT_DIR / "models" / "artifacts_research_v3_gpu_external"
MODEL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if BEST_MODEL_NAME not in final_fitted_models:
    raise RuntimeError("The validation-selected model is not available in final_fitted_models.")

best_full_pipeline = final_fitted_models[BEST_MODEL_NAME]
BEST_MODEL_INPUT_COLUMNS = list(X_train.columns)

best_model_artifact = {
    "artifact_name": "CIC-IDS2017 binary NIDS full-feature pipeline",
    "artifact_version": 3,
    "output_run_name": OUTPUT_RUN_NAME,
    "xgboost_device": XGB_DEVICE if BEST_MODEL_NAME == "XGBoost" else None,
    "model_name": BEST_MODEL_NAME,
    "pipeline": best_full_pipeline,
    "input_feature_columns": BEST_MODEL_INPUT_COLUMNS,
    "target_mapping": {"BENIGN": 0, "ALL_OTHER_NONEMPTY_LABELS": 1},
    "positive_class": 1,
    "training_dataset": "Original CIC-IDS2017 CSV files supplied by the user",
    "training_rows": int(len(X_train_validation)),
    "modeling_sample_rows": int(len(data)),
    "split_strategy": SPLIT_STRATEGY,
    "selected_hyperparameters": hyperparameter_searches[BEST_MODEL_NAME].best_params_,
    "selected_imbalance_and_preprocessing_pipeline": str(
        hyperparameter_searches[BEST_MODEL_NAME].best_estimator_.get_params(deep=False)
    ),
    "random_state": RANDOM_STATE,
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgb.__version__ if XGBOOST_AVAILABLE else None,
}

BEST_MODEL_ARTIFACT_PATH = MODEL_ARTIFACT_DIR / "v3_gpu_best_cicids2017_full_pipeline.joblib"
joblib.dump(best_model_artifact, BEST_MODEL_ARTIFACT_PATH, compress=3)

# Save human-readable metadata separately from the binary estimator.
metadata = {k: v for k, v in best_model_artifact.items() if k != "pipeline"}
metadata["selected_hyperparameters"] = repr(metadata["selected_hyperparameters"])
metadata_path = MODEL_ARTIFACT_DIR / "v3_gpu_best_cicids2017_full_pipeline_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=str)

# Also export the underlying native XGBoost model file where applicable.
model_step = best_full_pipeline.named_steps["model"]
if isinstance(model_step, FoldAwareXGBClassifier):
    native_model = model_step.estimator_
    native_path = MODEL_ARTIFACT_DIR / "v3_gpu_best_xgboost_estimator.json"
    native_model.save_model(str(native_path))
else:
    native_path = None

# Save the SHAP-reduced model as a complete inference bundle if available.
SHAP_REDUCED_ARTIFACT_PATH = None
if (
    BEST_TREE_NAME is not None
    and SELECTED_SHAP_K is not None
    and selected_reduced_features
    and shap_reduced_test_record is not None
):
    shap_reduced_bundle = {
        "artifact_name": "CIC-IDS2017 SHAP-reduced binary NIDS model",
        "model_name": BEST_TREE_NAME,
        "preprocessor": final_pre,
        "selector": final_selector,
        "selected_transformed_feature_indices": selected_idx,
        "selected_transformed_feature_names": list(selected_reduced_features),
        "raw_input_feature_columns": list(X_train.columns),
        "model": reduced_final_model,
        "target_mapping": {"BENIGN": 0, "ALL_OTHER_NONEMPTY_LABELS": 1},
        "positive_class": 1,
        "training_rows": int(len(X_train_validation)),
    }
    SHAP_REDUCED_ARTIFACT_PATH = MODEL_ARTIFACT_DIR / "v3_gpu_best_cicids2017_shap_reduced_bundle.joblib"
    joblib.dump(shap_reduced_bundle, SHAP_REDUCED_ARTIFACT_PATH, compress=3)

artifact_rows = [
    {"artifact": "Full-feature fitted pipeline", "path": str(BEST_MODEL_ARTIFACT_PATH), "created": BEST_MODEL_ARTIFACT_PATH.exists()},
    {"artifact": "Human-readable metadata", "path": str(metadata_path), "created": metadata_path.exists()},
]
if native_path is not None:
    artifact_rows.append({"artifact": "Native XGBoost JSON estimator", "path": str(native_path), "created": native_path.exists()})
if SHAP_REDUCED_ARTIFACT_PATH is not None:
    artifact_rows.append({"artifact": "SHAP-reduced inference bundle", "path": str(SHAP_REDUCED_ARTIFACT_PATH), "created": SHAP_REDUCED_ARTIFACT_PATH.exists()})
show_table(pd.DataFrame(artifact_rows), "Saved model artifacts")

# Demonstrate that the saved full pipeline can be loaded and used in this notebook session.
loaded_best_artifact = joblib.load(BEST_MODEL_ARTIFACT_PATH)
loaded_best_pipeline = loaded_best_artifact["pipeline"]
assert loaded_best_artifact["input_feature_columns"] == BEST_MODEL_INPUT_COLUMNS


def predict_with_saved_full_pipeline(artifact_path, X_new):
    """Load the saved end-to-end model and predict on a same-schema feature DataFrame."""
    artifact = joblib.load(artifact_path)
    expected = artifact["input_feature_columns"]
    missing = [column for column in expected if column not in X_new.columns]
    if missing:
        raise ValueError(
            f"Input dataset is missing {len(missing)} required model features: {missing[:20]}. "
            "Do not create placeholder columns; harmonize the schema or train on a common feature set."
        )
    X_aligned = X_new.loc[:, expected]
    predictions = artifact["pipeline"].predict(X_aligned)
    probabilities = artifact["pipeline"].predict_proba(X_aligned)[:, 1]
    return pd.DataFrame({
        "prediction": predictions,
        "prediction_label": np.where(predictions == 1, "Attack", "Benign"),
        "attack_probability": probabilities,
    }, index=X_new.index)


def predict_with_shap_reduced_bundle(artifact_path, X_new):
    """Inference helper for the optional SHAP-reduced bundle."""
    bundle = joblib.load(artifact_path)
    expected = bundle["raw_input_feature_columns"]
    missing = [column for column in expected if column not in X_new.columns]
    if missing:
        raise ValueError(f"Missing required raw input features: {missing[:20]}")
    X_raw = X_new.loc[:, expected]
    X_transformed = bundle["preprocessor"].transform(X_raw)
    selector = bundle["selector"]
    if selector != "passthrough":
        X_transformed = selector.transform(X_transformed)
    X_reduced = np.asarray(X_transformed)[:, bundle["selected_transformed_feature_indices"]]
    predictions = bundle["model"].predict(X_reduced)
    probabilities = bundle["model"].predict_proba(X_reduced)[:, 1]
    return pd.DataFrame({
        "prediction": predictions,
        "prediction_label": np.where(predictions == 1, "Attack", "Benign"),
        "attack_probability": probabilities,
    }, index=X_new.index)

# Small inference smoke test: verifies the saved object loads and accepts the expected schema.
artifact_smoke_predictions = predict_with_saved_full_pipeline(
    BEST_MODEL_ARTIFACT_PATH, X_test.iloc[:5]
)
show_table(artifact_smoke_predictions.reset_index(drop=True), "Saved-model inference smoke test (first five rows)")
print("Saved and reloaded:", BEST_MODEL_ARTIFACT_PATH)
print("Selected model:", loaded_best_artifact["model_name"])
print("Required input columns:", len(loaded_best_artifact["input_feature_columns"]))

## External generalization protocol — CSE-CIC-IDS2018

This evaluation is strictly **train on CIC-IDS2017, test on CSE-CIC-IDS2018**. External labels are used only after predictions to calculate metrics. The notebook:

1. Audits every CSV header and maps external feature names to the source model's canonical feature names.
2. Preserves the exact source feature order; it never fills absent semantic features with invented zero columns.
3. Excludes blank labels and repeated header rows (`Label`).
4. Uses the already-fitted source model when the full feature schema is compatible. If some source features are genuinely unavailable, it fits a separate common-feature pipeline using **CIC-IDS2017 train + validation only**.
5. Predicts CSE-CIC-IDS2018 in chunks and keeps compact metric arrays, avoiding a 5 GB all-rows DataFrame.
6. Reports aggregate metrics, confusion matrix, attack-family detection recall, scenario compatibility notes, and class-conditional feature drift.

The 2018 `Bot` label does not separate ARES from Zeus, so the notebook does not pretend to calculate separate rates for those implementations. A family marked "mixed / not separable" needs PCAP/log-level provenance for further splitting. Feature-drift results are diagnostics, not proof that a feature is semantically identical between CICFlowMeter versions.


## 28B. Cross-dataset generalization: CSE-CIC-IDS2018[¶](#28B.-Cross-dataset-generalization:-CSE-CIC-IDS2018)

### Why this dataset?[¶](#Why-this-dataset?)

Use **CSE-CIC-IDS2018**, specifically its **Processed Traffic Data for ML Algorithms** CSV files. It is a follow-on CIC/CSE network-intrusion dataset with labeled benign and malicious network flows and 80 CICFlowMeter-derived features. Its attack scenarios include brute force, Heartbleed, botnet, DoS/DDoS, web attacks, and infiltration. The official UNB page provides the data and AWS download instructions; COMIDDS documents the processed flow CSV structure and known dataset limitations.

- Official dataset and download instructions: <https://www.unb.ca/cic/datasets/ids-2018.html>
- Dataset documentation / processed feature example: <https://fkie-cad.github.io/COMIDDS/content/datasets/cse_cic_ids2018/>

**Label compatibility:** this project is binary. We map `Benign` to `0` and every other non-empty label to `1`. Therefore the external dataset has the same two target classes as the trained model, but its original attack-family names are not identical to the 2017 taxonomy. This is not a 15-class, label-for-label transfer experiment.

**Feature compatibility:** CIC-IDS2018 uses a related CICFlowMeter feature family, but headers differ (for example, `Dst Port` versus `Destination Port`, and `Tot Fwd Pkts` versus `Total Fwd Packets`). The code below harmonizes known aliases, checks the intersection, and trains a source-only model using the shared feature set. It does not fabricate missing columns or refit preprocessing on the external data. Feature-name agreement is necessary but not sufficient: CICFlowMeter versions, units, collection conditions, and labeling limitations can still cause domain shift.

### Local directory[¶](#Local-directory)

Download only the labeled processed CSV files, not the raw PCAP/log archive, into:

`data/external/CSE-CIC-IDS2018/Processed Traffic Data for ML Algorithms/`

The official page gives an AWS CLI command for the processed folder. Because the full dataset is large, the loader below performs a chunked label audit and draws a binary-stratified sample that preserves the external dataset's benign/attack prevalence. It then evaluates a model trained only on CIC-IDS2017. The external labels are used only for scoring, not for fitting, feature selection, threshold tuning, or model selection.

The comparison reports both (1) source CIC-IDS2017 test performance using the shared feature set and (2) external CSE-CIC-IDS2018 performance using the same trained model. This makes the observed cross-domain performance drop interpretable.

In [ ]:
# CSE-CIC-IDS2018: schema harmonization, streamed inference, attack-family analysis and feature drift.
# This section never fits or tunes on CSE-CIC-IDS2018. It only transforms and predicts with a
# model fitted on CIC-IDS2017. Rows are processed in chunks; no multi-million-row feature DataFrame
# is materialized in RAM.
EXTERNAL_DATASET_DIR = (
    PROJECT_DIR / "data" / "external" / "CSE-CIC-IDS2018"
    / "Processed Traffic Data for ML Algorithms"
)
EXTERNAL_MAX_ROWS = None            # None = all rows; a positive cap takes the first valid rows in sorted file order (not a stratified sample).
EXTERNAL_CHUNK_SIZE = 50_000        # Conservative for a laptop with 6 GB GPU memory.
MIN_EXTERNAL_FEATURE_OVERLAP = 0.90
FEATURE_DRIFT_SAMPLE_PER_CLASS = 20_000
RANDOM_EXTERNAL_SEED = RANDOM_STATE + 1808
SAVE_EXTERNAL_ROW_PREDICTIONS = False  # Set True only if you explicitly need a huge row-level file.


def normalize_schema_token(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).replace("\ufeff", "").casefold())


CIC2018_FEATURE_ALIASES = {
    "Destination Port": ["Dst Port"],
    "Total Fwd Packets": ["Tot Fwd Pkts"],
    "Total Backward Packets": ["Tot Bwd Pkts"],
    "Total Length of Fwd Packets": ["TotLen Fwd Pkts"],
    "Total Length of Bwd Packets": ["TotLen Bwd Pkts"],
    "Fwd Packet Length Max": ["Fwd Pkt Len Max"],
    "Fwd Packet Length Min": ["Fwd Pkt Len Min"],
    "Fwd Packet Length Mean": ["Fwd Pkt Len Mean"],
    "Fwd Packet Length Std": ["Fwd Pkt Len Std"],
    "Bwd Packet Length Max": ["Bwd Pkt Len Max"],
    "Bwd Packet Length Min": ["Bwd Pkt Len Min"],
    "Bwd Packet Length Mean": ["Bwd Pkt Len Mean"],
    "Bwd Packet Length Std": ["Bwd Pkt Len Std"],
    "Flow Bytes/s": ["Flow Byts/s"],
    "Flow Packets/s": ["Flow Pkts/s"],
    "Fwd IAT Total": ["Fwd IAT Tot"],
    "Bwd IAT Total": ["Bwd IAT Tot"],
    "Fwd Header Length": ["Fwd Header Len"],
    "Bwd Header Length": ["Bwd Header Len"],
    "Fwd Packets/s": ["Fwd Pkts/s"],
    "Bwd Packets/s": ["Bwd Pkts/s"],
    "Min Packet Length": ["Pkt Len Min"],
    "Max Packet Length": ["Pkt Len Max"],
    "Packet Length Mean": ["Pkt Len Mean"],
    "Packet Length Std": ["Pkt Len Std"],
    "Packet Length Variance": ["Pkt Len Var"],
    "FIN Flag Count": ["FIN Flag Cnt"],
    "SYN Flag Count": ["SYN Flag Cnt"],
    "RST Flag Count": ["RST Flag Cnt"],
    "PSH Flag Count": ["PSH Flag Cnt"],
    "ACK Flag Count": ["ACK Flag Cnt"],
    "URG Flag Count": ["URG Flag Cnt"],
    "ECE Flag Count": ["ECE Flag Cnt"],
    "Average Packet Size": ["Pkt Size Avg"],
    "Avg Fwd Segment Size": ["Fwd Seg Size Avg"],
    "Avg Bwd Segment Size": ["Bwd Seg Size Avg"],
    "Fwd Avg Bytes/Bulk": ["Fwd Byts/b Avg"],
    "Fwd Avg Packets/Bulk": ["Fwd Pkts/b Avg"],
    "Fwd Avg Bulk Rate": ["Fwd Blk Rate Avg"],
    "Bwd Avg Bytes/Bulk": ["Bwd Byts/b Avg"],
    "Bwd Avg Packets/Bulk": ["Bwd Pkts/b Avg"],
    "Bwd Avg Bulk Rate": ["Bwd Blk Rate Avg"],
    "Subflow Fwd Packets": ["Subflow Fwd Pkts"],
    "Subflow Fwd Bytes": ["Subflow Fwd Byts"],
    "Subflow Bwd Packets": ["Subflow Bwd Pkts"],
    "Subflow Bwd Bytes": ["Subflow Bwd Byts"],
    "Init_Win_bytes_forward": ["Init Fwd Win Byts"],
    "Init_Win_bytes_backward": ["Init Bwd Win Byts"],
    "act_data_pkt_fwd": ["Fwd Act Data Pkts"],
    "min_seg_size_forward": ["Fwd Seg Size Min"],
}


def read_external_csv_header(path):
    errors = []
    for enc in ["utf-8-sig", "cp1252", "latin1"]:
        try:
            columns = pd.read_csv(path, nrows=0, encoding=enc).columns.astype(str).tolist()  # preserve exact raw names for usecols
            if not columns:
                raise ValueError("Empty header")
            return columns, enc
        except Exception as exc:
            errors.append(f"{enc}: {exc}")
    raise RuntimeError(f"Cannot read header for {path.name}: {' | '.join(errors)}")


def map_external_features(raw_columns, required_features):
    lookup = {}
    for raw_col in raw_columns:
        lookup.setdefault(normalize_schema_token(raw_col), []).append(raw_col)
    mapping, missing, ambiguous = {}, [], {}
    for feature in required_features:
        tokens = [normalize_schema_token(feature)]
        tokens.extend(normalize_schema_token(x) for x in CIC2018_FEATURE_ALIASES.get(feature, []))
        matches = list(dict.fromkeys(raw for token in tokens for raw in lookup.get(token, [])))
        if len(matches) == 1:
            mapping[feature] = matches[0]
        elif not matches:
            missing.append(feature)
        else:
            ambiguous[feature] = matches
    return mapping, missing, ambiguous


def build_source_only_common_pipeline(source_pipeline, common_features):
    """Rebuild the source preprocessing on common columns, then fit on CIC-IDS2017 only."""
    source_pre = source_pipeline.named_steps["preprocessor"]
    new_transformers = []
    for name, transformer, columns in source_pre.transformers:
        if transformer == "drop":
            continue
        if isinstance(columns, str):
            chosen = [columns] if columns in common_features else []
        else:
            chosen = [c for c in list(columns) if c in common_features]
        if chosen:
            new_transformers.append((name, clone(transformer), chosen))
    if not new_transformers:
        raise ValueError("No source preprocessing branches remain after feature intersection.")
    common_pre = ColumnTransformer(
        new_transformers, remainder="drop", sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )
    common_pipe = ImbPipeline([
        ("preprocessor", common_pre),
        ("sampler", clone(source_pipeline.named_steps["sampler"])
         if source_pipeline.named_steps["sampler"] != "passthrough" else "passthrough"),
        ("selector", clone(source_pipeline.named_steps["selector"])
         if source_pipeline.named_steps["selector"] != "passthrough" else "passthrough"),
        ("model", clone(source_pipeline.named_steps["model"])),
    ])
    return common_pipe


def reservoir_update(existing, incoming, limit, rng):
    if incoming.empty or limit <= 0:
        return existing
    take = incoming.copy()
    take["_reservoir_key"] = rng.random(len(take))
    combined = pd.concat([existing, take], ignore_index=True)
    combined = combined.nsmallest(limit, "_reservoir_key").drop(columns="_reservoir_key")
    return combined.reset_index(drop=True)


def psi_from_reference(reference, comparison, bins=10):
    ref = np.asarray(reference, dtype=float)
    cur = np.asarray(comparison, dtype=float)
    ref, cur = ref[np.isfinite(ref)], cur[np.isfinite(cur)]
    if len(ref) < 20 or len(cur) < 20:
        return np.nan
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        lo, hi = float(np.min(ref)), float(np.max(ref))
        if lo == hi:
            return 0.0 if np.all(cur == lo) else np.nan
        edges = np.linspace(lo, hi, bins + 1)
    edges[0], edges[-1] = -np.inf, np.inf
    p = np.histogram(ref, bins=edges)[0].astype(float)
    q = np.histogram(cur, bins=edges)[0].astype(float)
    p = np.clip(p / max(1, p.sum()), 1e-6, None)
    q = np.clip(q / max(1, q.sum()), 1e-6, None)
    return float(np.sum((q - p) * np.log(q / p)))


if not EXTERNAL_DATASET_DIR.exists():
    print("CSE-CIC-IDS2018 not found; external section skipped:", EXTERNAL_DATASET_DIR)
else:
    external_csv_files = sorted(EXTERNAL_DATASET_DIR.rglob("*.csv"))
    if not external_csv_files:
        raise FileNotFoundError(f"No external CSV files found under {EXTERNAL_DATASET_DIR}")

    MODEL_INPUT_COLUMNS = list(X_train.columns)
    source_pipeline_for_external = final_fitted_models[BEST_MODEL_NAME]
    external_file_info, mappings_by_file = {}, {}
    schema_rows = []

    # Header-only pass: determine an unambiguous canonical feature map for every file.
    for path in external_csv_files:
        raw_columns, encoding = read_external_csv_header(path)
        label_matches = [c for c in raw_columns if normalize_schema_token(c) == "label"]
        if len(label_matches) != 1:
            raise ValueError(f"Expected one Label column in {path.name}; found {label_matches}")
        feature_map, missing, ambiguous = map_external_features(raw_columns, MODEL_INPUT_COLUMNS)
        if ambiguous:
            raise ValueError(f"Ambiguous feature mapping in {path.name}: {ambiguous}")
        duplicate_sources = pd.Series(list(feature_map.values())).value_counts()
        duplicate_sources = duplicate_sources[duplicate_sources > 1]
        if not duplicate_sources.empty:
            raise ValueError(f"One external column maps to multiple source features in {path.name}: {duplicate_sources.to_dict()}")
        external_file_info[path.name] = {
            "path": path, "encoding": encoding, "label_column": label_matches[0],
            "feature_map": feature_map,
        }
        mappings_by_file[path.name] = feature_map
        for feature in MODEL_INPUT_COLUMNS:
            schema_rows.append({
                "file": path.name, "source_feature_order": MODEL_INPUT_COLUMNS.index(feature),
                "source_feature": feature, "external_column": feature_map.get(feature),
                "available": feature in feature_map,
            })

    shared_features = [f for f in MODEL_INPUT_COLUMNS if all(f in mappings_by_file[p.name] for p in external_csv_files)]
    missing_anywhere = [f for f in MODEL_INPUT_COLUMNS if f not in shared_features]
    overlap_fraction = len(shared_features) / max(1, len(MODEL_INPUT_COLUMNS))
    external_schema_df = pd.DataFrame(schema_rows)
    show_table(external_schema_df, "CSE-CIC-IDS2018 schema map by file (source input order)", max_rows=150)
    save_table(external_schema_df, "v3_external_schema_mapping_by_file.csv")
    print(f"Common model features: {len(shared_features)}/{len(MODEL_INPUT_COLUMNS)} ({overlap_fraction:.1%})")
    if overlap_fraction < MIN_EXTERNAL_FEATURE_OVERLAP:
        raise ValueError("External feature overlap is below the configured minimum; inspect the schema mapping.")

    # Use the exact saved source model when its full input schema is available.
    # Otherwise train a separate model on CIC-IDS2017 train+validation using only the common columns.
    if shared_features == MODEL_INPUT_COLUMNS:
        external_evaluation_pipeline = source_pipeline_for_external
        external_pipeline_description = "Original CIC-IDS2017 final pipeline; exact complete feature set"
    else:
        external_evaluation_pipeline = build_source_only_common_pipeline(
            hyperparameter_searches[BEST_MODEL_NAME].best_estimator_, shared_features
        )
        print("Fitting a source-only common-feature model; no 2018 data is used for fitting.")
        external_evaluation_pipeline.fit(X_train_validation[shared_features], y_train_validation)
        external_pipeline_description = f"CIC-IDS2017-only refit on common feature intersection ({len(shared_features)} features)"

    # Verify the feature order is exactly the trained model's order before streaming.
    if list(shared_features) != [f for f in MODEL_INPUT_COLUMNS if f in shared_features]:
        raise AssertionError("Common feature order changed unexpectedly.")
    if CATEGORICAL_FEATURES:
        missing_cats = [c for c in CATEGORICAL_FEATURES if c in shared_features and c not in MODEL_INPUT_COLUMNS]
        if missing_cats:
            raise ValueError(f"Categorical features missing from model schema: {missing_cats}")

    # Scenario compatibility is documentation-derived, not inferred from the label alone.
    SCENARIO_COMPATIBILITY = {
        "ddos attack-hoic": ("additional implementation", "HOIC; not present as a named implementation in 2017"),
        "ddos attacks-loic-udp": ("additional implementation", "LOIC-UDP; not the 2017 LOIC-HTTP variant"),
        "ddos attacks-loic-http": ("matched implementation", "LOIC-HTTP counterpart in 2017"),
        "dos attacks-hulk": ("matched implementation", "Hulk counterpart in 2017"),
        "dos attacks-goldeneye": ("matched implementation", "GoldenEye counterpart in 2017"),
        "dos attacks-slowloris": ("matched implementation", "Slowloris counterpart in 2017"),
        "dos attacks-slowhttptest": ("matched implementation", "SlowHTTPTest counterpart in 2017"),
        "ftp-bruteforce": ("matched family/tool", "FTP-Patator counterpart in 2017"),
        "ssh-bruteforce": ("matched family/tool", "SSH-Patator counterpart in 2017"),
        "bot": ("mixed / not separable from CSV label", "2018 scenario includes ARES and Zeus; row-level tool provenance is not in this label"),
        "infilteration": ("modified scenario", "Infiltration activity and execution sequence differ"),
        "brute force -web": ("matched broad family", "Web brute-force family; compare original scenario details"),
        "brute force -xss": ("matched broad family", "XSS family; compare original scenario details"),
        "sql injection": ("matched broad family", "SQL injection family; very small support in 2018 export"),
    }

    # Select an operational threshold using CIC-IDS2017 validation only.
    # This is not fitted or optimized on external labels.
    if shared_features == MODEL_INPUT_COLUMNS:
        threshold_validation_pipeline = tuned_pipeline_by_model[BEST_MODEL_NAME]
    else:
        threshold_validation_pipeline = build_source_only_common_pipeline(
            hyperparameter_searches[BEST_MODEL_NAME].best_estimator_, shared_features
        )
        threshold_validation_pipeline.fit(X_train[shared_features], y_train)
    validation_prob_for_threshold = threshold_validation_pipeline.predict_proba(X_val[shared_features])[:, 1]
    threshold_candidates = np.linspace(0.05, 0.95, 19)
    threshold_rows = []
    for threshold_value in threshold_candidates:
        validation_threshold_pred = (validation_prob_for_threshold >= threshold_value).astype("int8")
        _tn, _fp, _fn, _tp = confusion_matrix(y_val, validation_threshold_pred, labels=[0, 1]).ravel()
        threshold_rows.append({
            "threshold": float(threshold_value),
            "f2_attack": fbeta_score(y_val, validation_threshold_pred, beta=2, zero_division=0),
            "precision_attack": precision_score(y_val, validation_threshold_pred, zero_division=0),
            "recall_attack": recall_score(y_val, validation_threshold_pred, zero_division=0),
            "false_positive_rate": _fp / (_fp + _tn) if (_fp + _tn) else np.nan,
        })
    threshold_search_df = pd.DataFrame(threshold_rows).sort_values(
        ["f2_attack", "precision_attack", "threshold"], ascending=[False, False, False]
    ).reset_index(drop=True)
    VALIDATION_SELECTED_THRESHOLD = float(threshold_search_df.iloc[0]["threshold"])
    show_table(threshold_search_df, "Decision-threshold selection on CIC-IDS2017 validation only", max_rows=30)
    print("Validation-selected F2 threshold:", VALIDATION_SELECTED_THRESHOLD)
    save_table(threshold_search_df, "v3_source_validation_threshold_sweep.csv")

    # Source-side reference samples for class-conditional feature drift.
    rng_external = np.random.default_rng(RANDOM_EXTERNAL_SEED)
    numeric_common = [f for f in shared_features if f in NUMERIC_FEATURES]
    source_drift_samples = {}
    for cls in [0, 1]:
        source_part = X_train_validation.loc[y_train_validation == cls, numeric_common]
        if len(source_part) > FEATURE_DRIFT_SAMPLE_PER_CLASS:
            source_part = source_part.sample(FEATURE_DRIFT_SAMPLE_PER_CLASS, random_state=RANDOM_EXTERNAL_SEED + cls)
        source_drift_samples[cls] = source_part.reset_index(drop=True)
    external_drift_reservoirs = {0: pd.DataFrame(columns=numeric_common), 1: pd.DataFrame(columns=numeric_common)}

    global_y_true, global_y_pred, global_y_pred_validation_threshold, global_y_prob = [], [], [], []
    label_counts = Counter()
    family_counts = {}
    conversion_audit = {}
    total_raw_rows = missing_or_blank_labels = repeated_header_rows = processed_rows = 0
    prediction_rows_to_save = []
    stop_all_files = False

    for file_idx, path in enumerate(external_csv_files, start=1):
        info = external_file_info[path.name]
        feature_map = info["feature_map"]
        available_in_file = [f for f in shared_features if f in feature_map]
        if available_in_file != shared_features:
            raise RuntimeError(f"The shared feature intersection changed for {path.name}.")
        raw_feature_columns = [feature_map[f] for f in shared_features]
        usecols = list(dict.fromkeys(raw_feature_columns + [info["label_column"]]))
        print(f"Streaming {file_idx}/{len(external_csv_files)}: {path.name}")

        for chunk in pd.read_csv(
            path, usecols=usecols, chunksize=EXTERNAL_CHUNK_SIZE,
            encoding=info["encoding"], low_memory=False,
        ):
            total_raw_rows += len(chunk)
            labels = chunk[info["label_column"]].astype("string").str.strip()
            is_header = labels.str.casefold().eq("label").fillna(False)
            repeated_header_rows += int(is_header.sum())
            valid = labels.notna() & labels.ne("") & ~is_header
            missing_mask = labels.isna() | labels.eq("").fillna(False)
            missing_or_blank_labels += int(missing_mask.sum())
            if not valid.any():
                continue
            if EXTERNAL_MAX_ROWS is not None:
                remaining = EXTERNAL_MAX_ROWS - processed_rows
                if remaining <= 0:
                    stop_all_files = True
                    break
                valid_idx = labels.index[valid][:remaining]
            else:
                valid_idx = labels.index[valid]
            if len(valid_idx) == 0:
                continue

            original_labels = labels.loc[valid_idx].astype(str)
            y_chunk = (~original_labels.str.casefold().eq("benign")).astype("int8").to_numpy()
            label_counts.update(original_labels.value_counts().to_dict())
            raw_X = chunk.loc[valid_idx, raw_feature_columns].copy()
            raw_X.columns = shared_features  # canonical model feature names AND exact source order

            # Match source dtypes; record parse failures and clean +/- infinity to NaN.
            for feature in shared_features:
                if feature in CATEGORICAL_FEATURES:
                    vals = raw_X[feature].astype("string").str.strip()
                    vals = vals.mask(vals.eq(""), pd.NA)
                    raw_X[feature] = pd.Series(vals.to_numpy(dtype=object, na_value=np.nan), index=raw_X.index)
                else:
                    raw = raw_X[feature].astype("string").str.strip().replace("", pd.NA)
                    original_missing = int(raw.isna().sum())
                    nonmissing = int(raw.notna().sum())
                    numeric = pd.to_numeric(raw, errors="coerce")
                    failed = nonmissing - int(numeric.notna().sum())
                    numeric_values = numeric.to_numpy(dtype=float, na_value=np.nan)
                    pos_inf = int(np.isposinf(numeric_values).sum())
                    neg_inf = int(np.isneginf(numeric_values).sum())
                    cleaned = numeric.replace([np.inf, -np.inf], np.nan)
                    audit = conversion_audit.setdefault(feature, {
                        "original_missing_or_blank": 0, "nonmissing_before_conversion": 0,
                        "parse_failures": 0, "positive_infinity": 0,
                        "negative_infinity": 0, "missing_after_numeric_cleaning": 0,
                    })
                    audit["original_missing_or_blank"] += original_missing
                    audit["nonmissing_before_conversion"] += nonmissing
                    audit["parse_failures"] += failed
                    audit["positive_infinity"] += pos_inf
                    audit["negative_infinity"] += neg_inf
                    audit["missing_after_numeric_cleaning"] += int(cleaned.isna().sum())
                    raw_X[feature] = cleaned.astype("float32")

            y_pred_chunk = external_evaluation_pipeline.predict(raw_X)
            y_prob_chunk = external_evaluation_pipeline.predict_proba(raw_X)[:, 1]
            y_pred_chunk = np.asarray(y_pred_chunk, dtype=np.int8)
            y_prob_chunk = np.asarray(y_prob_chunk, dtype=np.float32)
            y_pred_validation_threshold = (y_prob_chunk >= VALIDATION_SELECTED_THRESHOLD).astype("int8")

            global_y_true.append(y_chunk)
            global_y_pred.append(y_pred_chunk)
            global_y_pred_validation_threshold.append(y_pred_validation_threshold)
            global_y_prob.append(y_prob_chunk)
            processed_rows += len(y_chunk)

            # Label-level results: each original label is an attack family except Benign.
            for label in original_labels.unique():
                mask = original_labels.to_numpy() == label
                if str(label).casefold() == "benign":
                    continue
                item = family_counts.setdefault(str(label), {
                    "support": 0, "detected_default_050": 0,
                    "detected_validation_threshold": 0,
                })
                item["support"] += int(mask.sum())
                item["detected_default_050"] += int(y_pred_chunk[mask].sum())
                item["detected_validation_threshold"] += int(y_pred_validation_threshold[mask].sum())

            # Balanced reservoirs for class-conditional drift analysis, not model fitting.
            numeric_chunk = raw_X[numeric_common] if numeric_common else pd.DataFrame(index=raw_X.index)
            for cls in [0, 1]:
                class_part = numeric_chunk.loc[y_chunk == cls]
                external_drift_reservoirs[cls] = reservoir_update(
                    external_drift_reservoirs[cls], class_part,
                    FEATURE_DRIFT_SAMPLE_PER_CLASS, rng_external,
                )

            if SAVE_EXTERNAL_ROW_PREDICTIONS:
                # Write only the bounded sample in memory; users can enable a separate chunk writer if full output is needed.
                remaining_save = max(0, 200_000 - sum(len(x) for x in prediction_rows_to_save))
                if remaining_save:
                    keep = min(remaining_save, len(y_chunk))
                    prediction_rows_to_save.append(pd.DataFrame({
                        "external_label": original_labels.iloc[:keep].to_numpy(),
                        "y_true": y_chunk[:keep], "y_pred": y_pred_chunk[:keep],
                        "attack_probability": y_prob_chunk[:keep],
                    }))
            if EXTERNAL_MAX_ROWS is not None and processed_rows >= EXTERNAL_MAX_ROWS:
                stop_all_files = True
                break
        if stop_all_files:
            break

    if processed_rows == 0:
        raise ValueError("No valid labelled external rows were processed.")

    y_external_all = np.concatenate(global_y_true)
    pred_external_all = np.concatenate(global_y_pred)
    pred_external_validation_threshold_all = np.concatenate(global_y_pred_validation_threshold)
    prob_external_all = np.concatenate(global_y_prob)
    global_y_true.clear(); global_y_pred.clear(); global_y_pred_validation_threshold.clear(); global_y_prob.clear()
    external_metrics_default = calculate_metrics(y_external_all, pred_external_all, prob_external_all)
    external_metrics_validation_threshold = calculate_metrics(
        y_external_all, pred_external_validation_threshold_all, prob_external_all
    )
    source_common_prob = external_evaluation_pipeline.predict_proba(X_test[shared_features])[:, 1]
    source_common_pred = (source_common_prob >= 0.50).astype("int8")
    source_common_validation_threshold_pred = (source_common_prob >= VALIDATION_SELECTED_THRESHOLD).astype("int8")
    source_common_metrics = calculate_metrics(y_test, source_common_pred, source_common_prob)
    source_common_threshold_metrics = calculate_metrics(
        y_test, source_common_validation_threshold_pred, source_common_prob
    )

    external_summary_df = pd.DataFrame([
        {"evaluation_dataset": "CIC-IDS2017 internal test (threshold=0.50)", "rows": len(y_test), "threshold": 0.50, **source_common_metrics},
        {"evaluation_dataset": "CIC-IDS2017 internal test (validation-selected threshold)", "rows": len(y_test), "threshold": VALIDATION_SELECTED_THRESHOLD, **source_common_threshold_metrics},
        {"evaluation_dataset": "CSE-CIC-IDS2018 external (threshold=0.50)", "rows": processed_rows, "threshold": 0.50, **external_metrics_default},
        {"evaluation_dataset": "CSE-CIC-IDS2018 external (validation-selected threshold)", "rows": processed_rows, "threshold": VALIDATION_SELECTED_THRESHOLD, **external_metrics_validation_threshold},
    ])
    show_table(external_summary_df, "External generalization — full metric comparison", max_rows=10)
    save_table(external_summary_df, "v3_external_generalization_metrics.csv")

    external_label_counts_df = pd.DataFrame(
        [{"external_original_label": k, "rows": int(v)} for k, v in label_counts.most_common()]
    )
    show_table(external_label_counts_df, "Clean external label distribution", max_rows=50)
    save_table(external_label_counts_df, "v3_external_original_label_counts.csv")

    attack_family_rows = []
    for label, values in sorted(family_counts.items(), key=lambda kv: kv[0].casefold()):
        status, explanation = SCENARIO_COMPATIBILITY.get(
            label.casefold(), ("requires scenario-level documentation review", "Not confidently mapped"))
        support = values["support"]
        detected_default = values["detected_default_050"]
        detected_threshold = values["detected_validation_threshold"]
        attack_family_rows.append({
            "external_attack_label": label,
            "scenario_compatibility": status,
            "documentation_note": explanation,
            "support": support,
            "detected_at_050": detected_default,
            "missed_at_050": support - detected_default,
            "detection_recall_at_050": detected_default / support if support else np.nan,
            "detected_at_validation_threshold": detected_threshold,
            "missed_at_validation_threshold": support - detected_threshold,
            "detection_recall_at_validation_threshold": detected_threshold / support if support else np.nan,
        })
    external_attack_family_df = pd.DataFrame(attack_family_rows).sort_values(
        "attack_detection_recall", ascending=True
    )
    show_table(external_attack_family_df, "CSE-CIC-IDS2018 attack-family detection breakdown", max_rows=50)
    save_table(external_attack_family_df, "v3_external_attack_family_detection.csv")

    scenario_audit_rows = []
    for label, count in label_counts.items():
        status, explanation = SCENARIO_COMPATIBILITY.get(
            label.casefold(), ("requires scenario-level documentation review", "Not confidently mapped"))
        scenario_audit_rows.append({"external_label": label, "rows": int(count),
                                    "scenario_status": status, "documentation_note": explanation})
    scenario_audit_df = pd.DataFrame(scenario_audit_rows).sort_values("rows", ascending=False)
    save_table(scenario_audit_df, "v3_external_attack_scenario_compatibility.csv")

    conversion_audit_df_external = pd.DataFrame([
        {"feature": feature, **counts} for feature, counts in conversion_audit.items()
    ]).sort_values("parse_failures", ascending=False)
    if not conversion_audit_df_external.empty:
        show_table(conversion_audit_df_external.head(40), "External numeric conversion failures", max_rows=40)
        save_table(conversion_audit_df_external, "v3_external_numeric_conversion_audit.csv")

    # Feature drift, class-conditional (Benign compared with Benign; Attack compared with Attack).
    try:
        from scipy.stats import ks_2samp
        drift_rows = []
        for cls, class_name in [(0, "Benign"), (1, "Attack")]:
            src_sample = source_drift_samples[cls]
            ext_sample = external_drift_reservoirs[cls]
            for feature in numeric_common:
                a = pd.to_numeric(src_sample[feature], errors="coerce").to_numpy(dtype=float)
                b = pd.to_numeric(ext_sample[feature], errors="coerce").to_numpy(dtype=float)
                a, b = a[np.isfinite(a)], b[np.isfinite(b)]
                if len(a) < 20 or len(b) < 20:
                    continue
                ks = ks_2samp(a, b, alternative="two-sided", method="auto")
                drift_rows.append({
                    "class": class_name, "feature": feature,
                    "source_n": len(a), "external_n": len(b),
                    "source_median": float(np.median(a)), "external_median": float(np.median(b)),
                    "source_q01": float(np.quantile(a, 0.01)), "external_q01": float(np.quantile(b, 0.01)),
                    "source_q99": float(np.quantile(a, 0.99)), "external_q99": float(np.quantile(b, 0.99)),
                    "ks_statistic": float(ks.statistic), "ks_pvalue": float(ks.pvalue),
                    "population_stability_index": psi_from_reference(a, b),
                })
        feature_drift_df = pd.DataFrame(drift_rows)
        if not feature_drift_df.empty:
            feature_drift_df = feature_drift_df.sort_values(
                ["class", "ks_statistic"], ascending=[True, False]
            )
            show_table(feature_drift_df, "Class-conditional feature distribution drift", max_rows=80)
            save_table(feature_drift_df, "v3_external_feature_distribution_drift.csv")
    except Exception as exc:
        warnings.warn(f"Feature drift calculation was not completed: {exc!r}")

    # Confusion matrix and curves from the full external evaluation arrays.
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_external_all, pred_external_all, labels=[0, 1],
        display_labels=["Benign", "Attack"], values_format="d", cmap="Blues",
        ax=axes[0], colorbar=False,
    )
    axes[0].set_title("CSE-CIC-IDS2018 full confusion matrix")
    RocCurveDisplay.from_predictions(y_external_all, prob_external_all, name="CSE-CIC-IDS2018", ax=axes[1])
    axes[1].set_title("External ROC curve")
    PrecisionRecallDisplay.from_predictions(y_external_all, prob_external_all, name="CSE-CIC-IDS2018", ax=axes[2])
    axes[2].set_title("External precision–recall curve")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "v3_external_full_evaluation.png", dpi=180, bbox_inches="tight")
    plt.show()

    external_confusion = confusion_matrix(y_external_all, pred_external_all, labels=[0, 1])
    print("External rows physically scanned:", f"{total_raw_rows:,}")
    print("Repeated CSV header rows excluded:", f"{repeated_header_rows:,}")
    print("Missing/blank label rows excluded:", f"{missing_or_blank_labels:,}")
    print("Valid rows evaluated:", f"{processed_rows:,}")
    print("Pipeline used:", external_pipeline_description)
    print("External confusion matrix at threshold 0.50 [ [TN, FP], [FN, TP] ]:\n", external_confusion)
    print("External confusion matrix at validation-selected threshold [ [TN, FP], [FN, TP] ]:\n",
          confusion_matrix(y_external_all, pred_external_validation_threshold_all, labels=[0, 1]))
    if SAVE_EXTERNAL_ROW_PREDICTIONS and prediction_rows_to_save:
        external_prediction_sample_df = pd.concat(prediction_rows_to_save, ignore_index=True)
        save_table(external_prediction_sample_df, "v3_external_row_prediction_sample_first_200k.csv")

    external_model_artifact = {
        "artifact_name": "CIC-IDS2017 source-only pipeline for CSE-CIC-IDS2018 external evaluation",
        "artifact_version": 3,
        "model_name": BEST_MODEL_NAME,
        "pipeline": external_evaluation_pipeline,
        "input_feature_columns": shared_features,
        "external_column_mapping_by_file": mappings_by_file,
        "target_mapping": {"BENIGN": 0, "ALL_OTHER_NONEMPTY_NONHEADER_LABELS": 1},
        "source_dataset": "CIC-IDS2017",
        "external_evaluation_dataset": "CSE-CIC-IDS2018",
        "source_training_rows": int(len(X_train_validation)),
        "external_evaluated_rows": int(processed_rows),
        "validation_selected_threshold": VALIDATION_SELECTED_THRESHOLD,
        "threshold_selection_objective": "F2 on CIC-IDS2017 validation only",
        "pipeline_description": external_pipeline_description,
        "random_state": RANDOM_STATE,
        "xgboost_device": XGB_DEVICE if BEST_MODEL_NAME == "XGBoost" else None,
    }
    EXTERNAL_MODEL_ARTIFACT_PATH = MODEL_ARTIFACT_DIR / "v3_cicids2017_to_cicids2018_external_model.joblib"
    joblib.dump(external_model_artifact, EXTERNAL_MODEL_ARTIFACT_PATH, compress=3)
    print("Saved external-evaluation model bundle:", EXTERNAL_MODEL_ARTIFACT_PATH)


## 29. Experiment manifest and output inventory[¶](#29.-Experiment-manifest-and-output-inventory)

The manifest records the data source, sample settings, split protocol, CV protocol, selected model parameters, SHAP subset, and package versions. This makes the results auditable and supports the Methodology/Results sections of the paper.

In [ ]:
experiment_manifest = {
    "study": "CIC-IDS2017 binary NIDS: staged pipeline search, GPU XGBoost, complete split metrics, external generalization",
    "output_run_name": OUTPUT_RUN_NAME,
    "output_root": str(OUTPUT_ROOT),
    "model_artifact_dir": str(MODEL_ARTIFACT_DIR),
    "xgboost_device": XGB_DEVICE,
    "dataset_variant": "Original CIC-IDS2017 CSV files supplied by user",
    "raw_directory": str(RAW_DIR),
    "csv_files": [p.name for p in csv_files],
    "sampling_description": sampling_description,
    "max_rows_for_modeling": MAX_ROWS_FOR_MODELING,
    "search_max_rows": SEARCH_MAX_ROWS,
    "random_state": RANDOM_STATE,
    "split_strategy": SPLIT_STRATEGY,
    "cv_strategy": CV_STRATEGY,
    "cv_folds": CV_FOLDS,
    "target": "BENIGN=0; all other non-empty original labels=1",
    "models": MODEL_NAMES,
    "primary_metric": PRIMARY_SCORING,
    "method_catalogue_rows": len(METHOD_CATALOG),
    "best_validation_model": BEST_MODEL_NAME,
    "best_tree_model_for_shap": BEST_TREE_NAME,
    "best_imbalance_params": {m: repr(s.best_params_) for m, s in imbalance_searches.items()},
    "best_hyperparameters": {m: repr(s.best_params_) for m, s in hyperparameter_searches.items()},
    "combined_preprocessing_params": {m: repr(p) for m, p in combined_params_by_model.items()},
    "shap_selected_k": SELECTED_SHAP_K,
    "shap_selected_features": selected_reduced_features,
    "best_model_artifact_path": str(BEST_MODEL_ARTIFACT_PATH) if "BEST_MODEL_ARTIFACT_PATH" in globals() else None,
    "external_dataset_dir": str(EXTERNAL_DATASET_DIR) if "EXTERNAL_DATASET_DIR" in globals() else None,
    "external_generalization_evaluated": bool(
        "external_generalization_results_df" in globals()
        and not external_generalization_results_df.empty
    ),
    "cross_dataset_artifact_path": str(CROSS_DATASET_ARTIFACT_PATH) if "CROSS_DATASET_ARTIFACT_PATH" in globals() and CROSS_DATASET_ARTIFACT_PATH else None,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__ if XGBOOST_AVAILABLE else None,
    "shap": shap.__version__ if SHAP_AVAILABLE else None,
}
manifest_path = TABLES_DIR / "experiment_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(experiment_manifest, f, indent=2, default=str)

print("Tables:", TABLES_DIR)
print("Figures:", FIGURES_DIR)
print("Manifest:", manifest_path)

## 30. Research interpretation and reproducibility checklist[¶](#30.-Research-interpretation-and-reproducibility-checklist)

### Dataset and quality[¶](#Dataset-and-quality)

- Dataset version and all eight CSV inputs are identified.
- Per-file row counts and original attack-family counts are reported.
- Missing, infinite, duplicate, conflicting-vector, and outlier audits are reported.
- Cross-partition exact-vector overlap is reported.
- Any modeling sample is disclosed, including its sampling procedure.

### Experimental validity[¶](#Experimental-validity)

- Split protocol matches the stated generalization question.
- All learned preprocessing is fitted inside CV training folds.
- Variance filtering / feature selection is fitted inside CV training folds.
- Resampling is applied only inside CV training folds.
- Pipeline and hyperparameter searches do not use the test set.
- Validation is used for finalist comparison.
- The final test partition is evaluated only after all choices are frozen.

### Results and limitations[¶](#Results-and-limitations)

- Report CV means and standard deviations, preferably per-fold results too.
- Report Attack Precision/Recall/F1, Macro-F1, PR-AUC, ROC-AUC, FPR/FNR and MCC.
- Explain false positives and false negatives.
- Explain whether individually good preprocessing choices interact when combined.
- Compare full-feature and SHAP-reduced models.
- Report training time and prediction latency where feasible.
- Acknowledge CIC-IDS2017's documented quality limitations.
- Do not claim zero-day detection, cross-network generalization, or real-time deployment.
- Acknowledge that repeated validation-based search can overfit the validation set; the untouched test set is the final check.

### Model artifacts and external evaluation[¶](#Model-artifacts-and-external-evaluation)

- Save the validation-selected full-feature pipeline with preprocessing and input schema.
- Save the SHAP-reduced inference bundle when available.
- Preserve the exact model input-column order and binary target mapping.
- For CSE-CIC-IDS2018, report feature harmonization and use only shared, semantically matched features.
- Train the transfer model on CIC-IDS2017 only; do not refit or tune on external records.
- Report source-domain and external-domain scores for the same shared-feature model.
- State that binary labels match after mapping; original attack-family taxonomies do not match one-to-one.

## 31. Methodological references and source provenance[¶](#31.-Methodological-references-and-source-provenance)

### References used by the notebook[¶](#References-used-by-the-notebook)

- Scikit-learn, *Common pitfalls and recommended practices*: <https://scikit-learn.org/stable/common_pitfalls.html>
- Scikit-learn, *Pipeline and composite estimators*: <https://scikit-learn.org/stable/modules/compose.html>
- Scikit-learn, *GridSearchCV*: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html>
- Scikit-learn, *RandomizedSearchCV*: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html>
- Imbalanced-learn, *Pipeline*: <https://imbalanced-learn.org/stable/references/pipeline.html>
- SHAP, *TreeExplainer*: <https://shap.readthedocs.io/en/stable/generated/shap.TreeExplainer.html>
- Canadian Institute for Cybersecurity, *CIC-IDS2017*: <https://www.unb.ca/cic/datasets/ids-2017.html>
- Engelen, Rimmer and Joosen (2021), *Troubleshooting an Intrusion Detection Dataset: The CICIDS2017 Case Study*.
- Liu et al. (2022), *Error Prevalence in NIDS Datasets: A Case Study on CIC-IDS-2017 and CSE-CIC-IDS-2018*.

### Supplied framework[¶](#Supplied-framework)

The accompanying 36-page PDF is used as a method taxonomy and decision framework. Its generic pipeline options are not all valid search candidates for a low-cardinality Protocol feature or for this offline CIC-IDS2017 experiment. The notebook records which options are searched, conditional, deferred, visualization-only, or outside scope.

- Canadian Institute for Cybersecurity, *CSE-CIC-IDS2018*: <https://www.unb.ca/cic/datasets/ids-2018.html>
- COMIDDS, *CSE-CIC-IDS2018 dataset documentation*: <https://fkie-cad.github.io/COMIDDS/content/datasets/cse_cic_ids2018/>